In [2]:
import pandas as pd

# ground truth

In [3]:
data=pd.read_csv(r"C:\Users\King\Downloads\STEAM_GAMES.CSV")
data

,Unnamed: 0,game_name,app_id,review_text,review_length,hours_played,review_date,owners,developers,publishers,genres,platforms,categories,release_date,price
0,0,Counter-Strike: Global Offensive,730,pucajjj bam bam,15,197.216667,2026-03-02 01:55:35,"100,000,000 .. 200,000,000",['Valve'],['Valve'],"['Action', 'Free To Play']","['windows', 'linux']","['Multi-player', 'Cross-Platform Multiplayer',...","Aug 21, 2012",NaN
1,1,Counter-Strike: Global Offensive,730,YES,3,22.850000,2026-03-02 01:39:35,"100,000,000 .. 200,000,000",['Valve'],['Valve'],"['Action', 'Free To Play']","['windows', 'linux']","['Multi-player', 'Cross-Platform Multiplayer',...","Aug 21, 2012",NaN
2,2,Counter-Strike: Global Offensive,730,its pretty fun,14,24.150000,2026-03-02 01:32:54,"100,000,000 .. 200,000,000",['Valve'],['Valve'],"['Action', 'Free To Play']","['windows', 'linux']","['Multi-player', 'Cross-Platform Multiplayer',...","Aug 21, 2012",NaN
3,3,Counter-Strike: Global Offensive,730,awsone,6,264.666667,2026-03-02 01:30:30,"100,000,000 .. 200,000,000",['Valve'],['Valve'],"['Action', 'Free To Play']","['windows', 'linux']","['Multi-player', 'Cross-Platform Multiplayer',...","Aug 21, 2012",NaN
4,4,Counter-Strike: Global Offensive,730,s,1,136.316667,2026-03-02 01:30:21,"100,000,000 .. 200,000,000",['Valve'],['Valve'],"['Action', 'Free To Play']","['windows', 'linux']","['Multi-player', 'Cross-Platform Multiplayer',...","Aug 21, 2012",NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1995,1995,Wallpaper Engine,431960,Simply goated,13,3.616667,2026-02-28 21:36:51,"20,000,000 .. 50,000,000",['Wallpaper Engine Team'],['Wallpaper Engine Team'],"['Casual', 'Indie', 'Animation & Modeling', 'D...",['windows'],"['Steam Achievements', 'Steam Trading Cards', ...","Nov 16, 2018",499.0
1996,1996,Wallpaper Engine,431960,"""it cool""",9,0.183333,2026-02-28 21:30:32,"20,000,000 .. 50,000,000",['Wallpaper Engine Team'],['Wallpaper Engine Team'],"['Casual', 'Indie', 'Animation & Modeling', 'D...",['windows'],"['Steam Achievements', 'Steam Trading Cards', ...","Nov 16, 2018",499.0
1997,1997,Wallpaper Engine,431960,good\r\n,6,3.600000,2026-02-28 21:15:45,"20,000,000 .. 50,000,000",['Wallpaper Engine Team'],['Wallpaper Engine Team'],"['Casual', 'Indie', 'Animation & Modeling', 'D...",['windows'],"['Steam Achievements', 'Steam Trading Cards', ...","Nov 16, 2018",499.0
1998,1998,Wallpaper Engine,431960,i like cat,10,59.416667,2026-02-28 20:38:56,"20,000,000 .. 50,000,000",['Wallpaper Engine Team'],['Wallpaper Engine Team'],"['Casual', 'Indie', 'Animation & Modeling', 'D...",['windows'],"['Steam Achievements', 'Steam Trading Cards', ...","Nov 16, 2018",499.0


In [ ]:
from openai import OpenAI
import pandas as pd
import time
from tqdm import tqdm

tqdm.pandas()

# =========================
# GROQ CLIENT
# =========================
client = OpenAI(
    api_key="",
    base_url="https://api.groq.com/openai/v1"
)

# =========================
# SAMPLE DATA
# =========================
sample_df = data.sample(200, random_state=42).copy()

# =========================
# STRONG SENTIMENT PROMPT
# =========================
def get_sentiment_groq(text):

    prompt = f"""
You are a sentiment annotation system for Steam game reviews.

You MUST classify sentiment into exactly ONE label:

LABELS:
- positive
- negative
- neutral

========================
STRICT DECISION RULES:
========================

1. NEVER default to neutral.
2. neutral is ONLY for:
   - meaningless text (e.g. "engine", "ok", random words)
   - no opinion at all

3. If ANY emotion exists → MUST choose positive or negative.

4. Interpret gaming slang correctly:
   POSITIVE SIGNALS:
   fun, good, great, amazing, love, insane, addictive, enjoyable, awesome, nice

   NEGATIVE SIGNALS:
   lag, bug, crash, broken, trash, bad, boring, unplayable, pay-to-win, worst

5. Even VERY short text MUST be classified if possible.

6. If mixed sentiment exists → choose the dominant one.

========================
EXAMPLES:
========================
"this game is fun" → positive
"full of bugs and lag" → negative
"i love it" → positive
"trash game" → negative
"engine" → neutral
"not bad actually fun" → positive
"game is ok" → neutral

========================
REVIEW:
{text}
========================

Return ONLY one word:
positive OR negative OR neutral
"""

    try:
        response = client.chat.completions.create(
            model="llama-3.1-8b-instant",
            messages=[
                {"role": "user", "content": prompt}
            ],
            temperature=0,
            max_tokens=5
        )

        time.sleep(0.2)  # reduced delay for speed

        label = response.choices[0].message.content.strip().lower()

        # safety cleanup (very important)
        if "positive" in label:
            return "positive"
        elif "negative" in label:
            return "negative"
        elif "neutral" in label:
            return "neutral"
        else:
            return "neutral"

    except Exception as e:
        print("Error:", e)
        return "neutral"




In [ ]:
from openai import OpenAI
import time

mistral_client = OpenAI(
    api_key="",
    base_url="https://api.mistral.ai/v1"
)

def get_sentiment_mistral_small(text):
    prompt = f"""
You are an expert sentiment annotator for Steam game reviews.

Classify into EXACTLY one label:
positive
negative
neutral

Rules:
- fun/love/amazing = positive
- bugs/crash/lag = negative
- neutral only if no opinion
- not bad = positive
- mixed = dominant sentiment

Review:
{text}

Return one word only.
"""

    try:
        response = mistral_client.chat.completions.create(
            model="mistral-small-latest",
            messages=[
                {"role": "user", "content": prompt}
            ],
            temperature=0,
            max_tokens=5
        )

        time.sleep(1)  # Mistral free = 1 req/sec

        label = response.choices[0].message.content.strip().lower()

        if "positive" in label:
            return "positive"
        elif "negative" in label:
            return "negative"
        elif "neutral" in label:
            return "neutral"
        else:
            return "neutral"

    except Exception as e:
        print("Mistral Error:", e)
        return "neutral"

In [ ]:
from openai import OpenAI

mistral_client = OpenAI(
    api_key="",
    base_url="https://api.mistral.ai/v1"
)

def get_sentiment_mistral_large(text):
    prompt = f"""
You are a sentiment classifier for Steam game reviews.

Classify the review into EXACTLY one label:
positive
negative
neutral

Rules:
- fun, amazing, love → positive
- bugs, crash, lag → negative
- no opinion → neutral
- mixed → dominant sentiment

Return ONLY one word.

Review:
{text}
"""

    try:
        response = mistral_client.chat.completions.create(
            model="mistral-large-latest",
            messages=[
                {"role": "user", "content": prompt}
            ],
            temperature=0,
            max_tokens=5
        )

        time.sleep(1)  # rate limit safety

        label = response.choices[0].message.content.strip().lower()

        if "positive" in label:
            return "positive"
        elif "negative" in label:
            return "negative"
        elif "neutral" in label:
            return "neutral"
        else:
            return "neutral"

    except Exception as e:
        print("mistral Error:", e)
        return "neutral"

In [40]:
from tqdm import tqdm
tqdm.pandas()

# sample
sample_df = data.sample(200, random_state=42).copy()

# # mistral samall
sample_df["label_1"] = sample_df["review_text"].progress_apply(get_sentiment_mistral_small)

# mistral large
sample_df["label_2"] = sample_df["review_text"].progress_apply(get_sentiment_mistral_large)

# groq llama
sample_df["label_3"] = sample_df["review_text"].progress_apply(get_sentiment_groq)


sample_df

100%|██████████| 200/200 [10:17<00:00,  3.09s/it]


,Unnamed: 0,game_name,app_id,review_text,review_length,hours_played,review_date,owners,developers,publishers,genres,platforms,categories,release_date,price,label_1,label_2,label_3
1860,1860,Path of Exile 2,2694490,`,1,10.600000,2026-02-26 23:50:27,"20,000,000 .. 50,000,000",['Grinding Gear Games'],['Grinding Gear Games'],"['Action', 'Adventure', 'Massively Multiplayer...",['windows'],"['Single-player', 'Multi-player', 'MMO', 'Co-o...","Dec 6, 2024",2999.0,neutral,neutral,neutral
353,353,Palworld,1623730,incredibly fun and great replayability (i thin...,79,92.016667,2026-02-28 14:33:41,"50,000,000 .. 100,000,000",['Pocketpair'],['Pocketpair'],"['Action', 'Adventure', 'Indie', 'RPG', 'Early...",['windows'],"['Single-player', 'Multi-player', 'Co-op', 'On...","Jan 18, 2024",2999.0,positive,positive,positive
1333,1333,Monster Hunter Wilds,2246340,I Monster my Hunter till I Wilds,32,113.883333,2026-03-01 10:51:44,"20,000,000 .. 50,000,000","['CAPCOM Co., Ltd.']","['CAPCOM Co., Ltd.']","['Action', 'Adventure', 'RPG']",['windows'],"['Single-player', 'Multi-player', 'Co-op', 'On...","Feb 27, 2025",6999.0,neutral,neutral,negative
905,905,Left 4 Dead 2,550,This is a really good game if you love Zombie ...,74,33.550000,2026-03-01 23:18:17,"50,000,000 .. 100,000,000",['Valve'],['Valve'],['Action'],"['windows', 'linux']","['Single-player', 'Multi-player', 'PvP', 'Onli...","Nov 16, 2009",999.0,positive,positive,positive
1289,1289,War Thunder,236390,love the game but it pisses me the ♥♥♥♥ off un...,65,224.633333,2024-01-15 07:27:11,"20,000,000 .. 50,000,000",['Gaijin Entertainment'],['Gaijin Network Ltd'],"['Action', 'Massively Multiplayer', 'Simulatio...","['windows', 'mac', 'linux']","['Single-player', 'Multi-player', 'MMO', 'PvP'...","Aug 15, 2013",NaN,positive,negative,negative
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
462,462,Team Fortress 2,440,the OG. The big chunga. This was one of the bi...,94,103.883333,2026-02-20 16:00:58,"50,000,000 .. 100,000,000",['Valve'],['Valve'],"['Action', 'Free To Play']","['windows', 'linux']","['Multi-player', 'Cross-Platform Multiplayer',...","Oct 10, 2007",NaN,positive,positive,positive
1105,1105,Lost Ark,1599340,I'm so grateful I quit playing in June of 2022...,319,924.116667,2026-02-22 04:17:29,"50,000,000 .. 100,000,000",['Smilegate RPG'],['Amazon Game Studios'],"['Action', 'Adventure', 'Massively Multiplayer...",['windows'],"['Single-player', 'Multi-player', 'MMO', 'PvP'...","Feb 11, 2022",NaN,neutral,negative,negative
855,855,Grand Theft Auto V Legacy,271590,best game,9,31.683333,2026-03-01 07:10:12,"50,000,000 .. 100,000,000",['Rockstar North'],['Rockstar Games'],"['Action', 'Adventure']",['windows'],"['Single-player', 'Multi-player', 'PvP', 'Onli...","Apr 13, 2015",NaN,positive,positive,positive
693,693,New World: Aeternum,1063730,I started a few times over since the story cha...,592,1342.783333,2026-02-08 09:09:30,"50,000,000 .. 100,000,000",['Amazon Game Studios'],['Amazon Game Studios'],"['Action', 'Adventure', 'Massively Multiplayer...",['windows'],"['Multi-player', 'MMO', 'PvP', 'Online PvP', '...","Sep 28, 2021",NaN,negative,negative,negative


In [41]:
sample_df["label_1"].value_counts()

label_1
positive    119
negative     43
neutral      38
Name: count, dtype: int64

In [42]:
sample_df["label_2"].value_counts()

label_2
positive    117
negative     50
neutral      33
Name: count, dtype: int64

In [43]:
sample_df["label_3"].value_counts()

label_3
positive    107
negative     63
neutral      30
Name: count, dtype: int64

In [44]:
sample_df

,Unnamed: 0,game_name,app_id,review_text,review_length,hours_played,review_date,owners,developers,publishers,genres,platforms,categories,release_date,price,label_1,label_2,label_3
1860,1860,Path of Exile 2,2694490,`,1,10.600000,2026-02-26 23:50:27,"20,000,000 .. 50,000,000",['Grinding Gear Games'],['Grinding Gear Games'],"['Action', 'Adventure', 'Massively Multiplayer...",['windows'],"['Single-player', 'Multi-player', 'MMO', 'Co-o...","Dec 6, 2024",2999.0,neutral,neutral,neutral
353,353,Palworld,1623730,incredibly fun and great replayability (i thin...,79,92.016667,2026-02-28 14:33:41,"50,000,000 .. 100,000,000",['Pocketpair'],['Pocketpair'],"['Action', 'Adventure', 'Indie', 'RPG', 'Early...",['windows'],"['Single-player', 'Multi-player', 'Co-op', 'On...","Jan 18, 2024",2999.0,positive,positive,positive
1333,1333,Monster Hunter Wilds,2246340,I Monster my Hunter till I Wilds,32,113.883333,2026-03-01 10:51:44,"20,000,000 .. 50,000,000","['CAPCOM Co., Ltd.']","['CAPCOM Co., Ltd.']","['Action', 'Adventure', 'RPG']",['windows'],"['Single-player', 'Multi-player', 'Co-op', 'On...","Feb 27, 2025",6999.0,neutral,neutral,negative
905,905,Left 4 Dead 2,550,This is a really good game if you love Zombie ...,74,33.550000,2026-03-01 23:18:17,"50,000,000 .. 100,000,000",['Valve'],['Valve'],['Action'],"['windows', 'linux']","['Single-player', 'Multi-player', 'PvP', 'Onli...","Nov 16, 2009",999.0,positive,positive,positive
1289,1289,War Thunder,236390,love the game but it pisses me the ♥♥♥♥ off un...,65,224.633333,2024-01-15 07:27:11,"20,000,000 .. 50,000,000",['Gaijin Entertainment'],['Gaijin Network Ltd'],"['Action', 'Massively Multiplayer', 'Simulatio...","['windows', 'mac', 'linux']","['Single-player', 'Multi-player', 'MMO', 'PvP'...","Aug 15, 2013",NaN,positive,negative,negative
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
462,462,Team Fortress 2,440,the OG. The big chunga. This was one of the bi...,94,103.883333,2026-02-20 16:00:58,"50,000,000 .. 100,000,000",['Valve'],['Valve'],"['Action', 'Free To Play']","['windows', 'linux']","['Multi-player', 'Cross-Platform Multiplayer',...","Oct 10, 2007",NaN,positive,positive,positive
1105,1105,Lost Ark,1599340,I'm so grateful I quit playing in June of 2022...,319,924.116667,2026-02-22 04:17:29,"50,000,000 .. 100,000,000",['Smilegate RPG'],['Amazon Game Studios'],"['Action', 'Adventure', 'Massively Multiplayer...",['windows'],"['Single-player', 'Multi-player', 'MMO', 'PvP'...","Feb 11, 2022",NaN,neutral,negative,negative
855,855,Grand Theft Auto V Legacy,271590,best game,9,31.683333,2026-03-01 07:10:12,"50,000,000 .. 100,000,000",['Rockstar North'],['Rockstar Games'],"['Action', 'Adventure']",['windows'],"['Single-player', 'Multi-player', 'PvP', 'Onli...","Apr 13, 2015",NaN,positive,positive,positive
693,693,New World: Aeternum,1063730,I started a few times over since the story cha...,592,1342.783333,2026-02-08 09:09:30,"50,000,000 .. 100,000,000",['Amazon Game Studios'],['Amazon Game Studios'],"['Action', 'Adventure', 'Massively Multiplayer...",['windows'],"['Multi-player', 'MMO', 'PvP', 'Online PvP', '...","Sep 28, 2021",NaN,negative,negative,negative


In [45]:
from statsmodels.stats.inter_rater import fleiss_kappa
import numpy as np
import pandas as pd

# labels dataframe
df = sample_df[["label_1", "label_2", "label_3"]]

def to_fleiss(row):
    return [
        (row == "positive").sum(),
        (row == "negative").sum(),
        (row == "neutral").sum()
    ]

fleiss_matrix = df.apply(to_fleiss, axis=1).tolist()


fleiss_matrix = np.array(fleiss_matrix)

score = fleiss_kappa(fleiss_matrix)

print("Fleiss Kappa =", score)

Fleiss Kappa = 0.7776858152001309


In [46]:
labels = sample_df[["label_1", "label_2", "label_3"]]

def get_final_label(row):
    counts = row.value_counts()

    # majority vote
    if len(counts) > 0:
        top_label = counts.idxmax()

        
        if list(counts.values).count(counts.max()) > 1:
            return row["label_3"]  # trust mistral large as fallback

        return top_label

    return "neutral"

sample_df["final_label"] = labels.apply(get_final_label, axis=1)


In [49]:
sample_df[sample_df["final_label"]=="negative"]["review_text"].to_frame().sample(5)

,review_text
585,its bad on steam only I've been facing an issu...
1244,vary skill based with a vary high learning car...
1289,love the game but it pisses me the ♥♥♥♥ off un...
602,"Game is dead now, but the leveling felt good, ..."
1818,Mid game boring asf


In [53]:
sample_df[sample_df["final_label"]=="positive"]["review_text"].to_frame().sample(5)

,review_text
824,One of the best games OAT could play for an et...
1532,Yes it is a good game there is so much to do.
1609,"Margit, the perfect teacher!"
1666,"Fantastic Game, can get a little difficult at ..."
987,I love this game and it still is very good and...


In [54]:
sample_df.to_csv(r"D:\\final_ground_truth.CSV")

# design 3 schemas

In [3]:
pip install deep-translator

Defaulting to user installation because normal site-packages is not writeable
  Using cached deep_translator-1.11.4-py3-none-any.whl.metadata (30 kB)
Using cached deep_translator-1.11.4-py3-none-any.whl (42 kB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
!pip install emoji

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
import pandas as pd
import re
from textblob import Word, TextBlob
from symspellpy.symspellpy import SymSpell, Verbosity
from tqdm.notebook import tqdm
import emoji
from deep_translator import GoogleTranslator
from nltk.corpus import stopwords
import nltk
import pkg_resources

nltk.download('stopwords', quiet=True)
# tqdm.pandas()


# Initialize SymSpell
sym_spell = SymSpell(max_dictionary_edit_distance=2)
dictionary_path = pkg_resources.resource_filename(
    "symspellpy", "frequency_dictionary_en_82_765.txt"
)
sym_spell.load_dictionary(dictionary_path, 0, 1)

print("Setup complete.")

Setup complete.


In [6]:
def translate_to_english(text):
    if pd.isna(text) or str(text).strip() == "":
        return text
    try:
        return GoogleTranslator(source='auto', target='en').translate(str(text))
    except:
        return text

def fix_encoding_fn(text):
    if pd.isna(text):
        return text
    return text.encode("ascii", "ignore").decode()

def remove_noise_fn(text):
    if pd.isna(text):
        return text
    text = re.sub(r"\.{2,}", " ", text)
    text = re.sub(r"[^a-zA-Z0-9\s]", " ", text)
    return text.strip()

def lowercase_text(text):
    if pd.isna(text):
        return text
    return text.lower()

def remove_numbers_fn(text):
    if pd.isna(text):
        return text
    return re.sub(r"\d+", "", text)

def fix_spelling_fn(text):   # <-- was: def fix_spelling(text)
    if pd.isna(text):
        return text
    words = text.split()
    corrected = []
    for word in words:
        suggestions = sym_spell.lookup(word, Verbosity.CLOSEST, max_edit_distance=2)
        if suggestions:
            corrected.append(suggestions[0].term)
        else:
            corrected.append(str(TextBlob(word).correct()))
    return " ".join(corrected)

def lemmatize_text(text):
    if pd.isna(text):
        return text
    words = text.split()
    lemmas = [Word(w).lemmatize() for w in words]
    return " ".join(lemmas)

def remove_stopwords_fn(text):
    if pd.isna(text):
        return text
    opinion_words = {
        'no', 'not', 'nor', 'never', 'neither', 'nobody', 'nothing',
        'nowhere', 'hardly', 'scarcely', 'barely', "don't", "doesn't",
        "didn't", "won't", "wouldn't", "shouldn't", "couldn't", "isn't",
        "aren't", "wasn't", "weren't"
    }
    stop_words = set(stopwords.words('english')) - opinion_words
    words = text.split()
    return " ".join([w for w in words if w.lower() not in stop_words])

def remove_emojis_fn(text):
    if pd.isna(text):
        return text
    return emoji.replace_emoji(text, replace='').strip()

def extract_genre(text):
    if pd.isna(text):
        return "Unknown"
    match = re.search(r"'(.*?)'", text)
    return match.group(1) if match else "Unknown"

print("Functions defined.")

Functions defined.


In [7]:
def run_pipeline(
    input_file,
    output_file,
    text_column="review_text",
    category_column="genres",
    translate=False,
    fix_encoding=False,
    remove_noise=False,
    lowercase=False,
    remove_numbers=False,
    fix_spelling=False,
    lemmatize=False,
    extract_tags=False,
    remove_stopwords=False,
    remove_emojis=False,
):
   
    print(f"\n{'='*60}")
    print(f"  INPUT : {input_file}")
    print(f"  OUTPUT: {output_file}")
    print(f"{'='*60}")

    df = pd.read_csv(input_file)
    print(f"Loaded {len(df):,} rows.\n")

    # Convert text column to object to avoid pyarrow issues
    df[text_column] = df[text_column].astype(object)

    # Ordered steps: (flag, label, function)
    steps = [
    (translate,        "Translating to English...",    translate_to_english),
    (fix_encoding,     "Fixing encoding artifacts...", fix_encoding_fn),
    (remove_noise,     "Removing noise...",            remove_noise_fn),
    (lowercase,        "Converting to lowercase...",   lowercase_text),
    (remove_numbers,   "Removing numbers...",          remove_numbers_fn),
    (fix_spelling,     "Fixing spelling...",           fix_spelling_fn),   # <-- _fn added
    (lemmatize,        "Applying lemmatization...",    lemmatize_text),
    (remove_stopwords, "Removing stopwords...",        remove_stopwords_fn),
    (remove_emojis,    "Removing emojis...",           remove_emojis_fn),
    ]

    for flag, label, fn in steps:
        if flag:
            print(label)
            df[text_column] = df[text_column].apply(fn)

    if extract_tags:
        print("Extracting category tags...")
        df['category'] = df[category_column].apply(extract_genre)

    print(f"\n--- Summary ---")
    print(f"Rows             : {len(df):,}")
    print(f"Empty text rows  : {df[text_column].isna().sum()}")
    if extract_tags and 'category' in df.columns:
        print(f"Unique categories: {df['category'].nunique()}")

    df.to_csv(output_file, index=False)
    print(f"\nSaved -> {output_file}")

    return df

print("run_pipeline() ready.")

run_pipeline() ready.


In [9]:
df_gt = run_pipeline(
    input_file       = r"D:\final_ground_truth.CSV",
    output_file      = "STEAM_GAMES_CLEAN_GTLabel.csv",
    translate        = True,
    fix_encoding     = True,
    remove_noise     = True,
    lowercase        = True,
    remove_numbers   = True,
    fix_spelling     = True,
    lemmatize        = True,
    extract_tags     = True,
    remove_stopwords = True,
    remove_emojis    = True,
)
df_gt.head()


  INPUT : D:\final_ground_truth.CSV
  OUTPUT: STEAM_GAMES_CLEAN_GTLabel.csv
Loaded 200 rows.

Translating to English...
Fixing encoding artifacts...
Removing noise...
Converting to lowercase...
Removing numbers...
Fixing spelling...
Applying lemmatization...
Removing stopwords...
Removing emojis...
Extracting category tags...

--- Summary ---
Rows             : 200
Empty text rows  : 6
Unique categories: 2

Saved -> STEAM_GAMES_CLEAN_GTLabel.csv


,Unnamed: 0.1,Unnamed: 0,game_name,app_id,review_text,review_length,hours_played,review_date,owners,developers,...,genres,platforms,categories,release_date,price,label_1,label_2,label_3,final_label,category
0,1860,1860,Path of Exile 2,2694490,None,1,10.600000,2026-02-26 23:50:27,"20,000,000 .. 50,000,000",['Grinding Gear Games'],...,"['Action', 'Adventure', 'Massively Multiplayer...",['windows'],"['Single-player', 'Multi-player', 'MMO', 'Co-o...","Dec 6, 2024",2999.0,neutral,neutral,neutral,neutral,Action
1,353,353,Palworld,1623730,incredibly fun great repeatability think spell...,79,92.016667,2026-02-28 14:33:41,"50,000,000 .. 100,000,000",['Pocketpair'],...,"['Action', 'Adventure', 'Indie', 'RPG', 'Early...",['windows'],"['Single-player', 'Multi-player', 'Co-op', 'On...","Jan 18, 2024",2999.0,positive,positive,positive,positive,Action
2,1333,1333,Monster Hunter Wilds,2246340,monster hunter till wild,32,113.883333,2026-03-01 10:51:44,"20,000,000 .. 50,000,000","['CAPCOM Co., Ltd.']",...,"['Action', 'Adventure', 'RPG']",['windows'],"['Single-player', 'Multi-player', 'Co-op', 'On...","Feb 27, 2025",6999.0,neutral,neutral,negative,neutral,Action
3,905,905,Left 4 Dead 2,550,really good game love zombie apocalypse game g...,74,33.550000,2026-03-01 23:18:17,"50,000,000 .. 100,000,000",['Valve'],...,['Action'],"['windows', 'linux']","['Single-player', 'Multi-player', 'PvP', 'Onli...","Nov 16, 2009",999.0,positive,positive,positive,positive,Action
4,1289,1289,War Thunder,236390,love game piss unlike game,65,224.633333,2024-01-15 07:27:11,"20,000,000 .. 50,000,000",['Gaijin Entertainment'],...,"['Action', 'Massively Multiplayer', 'Simulatio...","['windows', 'mac', 'linux']","['Single-player', 'Multi-player', 'MMO', 'PvP'...","Aug 15, 2013",NaN,positive,negative,negative,negative,Action


In [10]:
df_schema1 = run_pipeline(
    input_file   = r"D:\chrom download\STEAM_GAMES_REDUCED.csv",
    output_file  = "SCHEMA_1.csv",
    translate    = True,
    fix_encoding = True,
    remove_noise = True,
    lowercase    = True,
)
df_schema1.head()


  INPUT : D:\chrom download\STEAM_GAMES_REDUCED.csv
  OUTPUT: SCHEMA_1.csv
Loaded 200 rows.

Translating to English...
Fixing encoding artifacts...
Removing noise...
Converting to lowercase...

--- Summary ---
Rows             : 200
Empty text rows  : 2

Saved -> SCHEMA_1.csv


,Unnamed: 0,game_name,app_id,review_text,review_length,hours_played,review_date,owners,developers,publishers,genres,platforms,categories,release_date,price
0,83,Counter-Strike: Global Offensive,730,yes very well my eyes are gone after the flash...,52,23.950000,2026-03-01 22:35:30,"100,000,000 .. 200,000,000",['Valve'],['Valve'],"['Action', 'Free To Play']","['windows', 'linux']","['Multi-player', 'Cross-Platform Multiplayer',...","Aug 21, 2012",NaN
1,53,Counter-Strike: Global Offensive,730,as awesome as always but please bring the oth...,85,209.766667,2026-03-01 23:39:12,"100,000,000 .. 200,000,000",['Valve'],['Valve'],"['Action', 'Free To Play']","['windows', 'linux']","['Multi-player', 'Cross-Platform Multiplayer',...","Aug 21, 2012",NaN
2,70,Counter-Strike: Global Offensive,730,toooooooo good,14,27.500000,2026-03-01 23:00:25,"100,000,000 .. 200,000,000",['Valve'],['Valve'],"['Action', 'Free To Play']","['windows', 'linux']","['Multi-player', 'Cross-Platform Multiplayer',...","Aug 21, 2012",NaN
3,45,Counter-Strike: Global Offensive,730,counter strike is the best game in the world a...,88,114.666667,2026-03-02 00:02:24,"100,000,000 .. 200,000,000",['Valve'],['Valve'],"['Action', 'Free To Play']","['windows', 'linux']","['Multi-player', 'Cross-Platform Multiplayer',...","Aug 21, 2012",NaN
4,44,Counter-Strike: Global Offensive,730,game 3ys volvo,19,855.216667,2026-03-02 00:05:50,"100,000,000 .. 200,000,000",['Valve'],['Valve'],"['Action', 'Free To Play']","['windows', 'linux']","['Multi-player', 'Cross-Platform Multiplayer',...","Aug 21, 2012",NaN


In [12]:
df_schema2 = run_pipeline(
    input_file     = r"C:\Users\King\Downloads\SCHEMA_1.csv",
    output_file    = "SCHEMA_2.csv",
    remove_numbers = True,
    remove_emojis  = True,
    lemmatize      = True,
)
df_schema2.head()


  INPUT : C:\Users\King\Downloads\SCHEMA_1.csv
  OUTPUT: SCHEMA_2.csv
Loaded 200 rows.

Removing numbers...
Applying lemmatization...
Removing emojis...

--- Summary ---
Rows             : 200
Empty text rows  : 6

Saved -> SCHEMA_2.csv


,Unnamed: 0.2,Unnamed: 0.1,Unnamed: 0,game_name,app_id,review_text,review_length,hours_played,review_date,owners,developers,publishers,genres,platforms,categories,release_date,price
0,0,1860,1860,Path of Exile 2,2694490,NaN,1,10.600000,2026-02-26 23:50:27,"20,000,000 .. 50,000,000",['Grinding Gear Games'],['Grinding Gear Games'],"['Action', 'Adventure', 'Massively Multiplayer...",['windows'],"['Single-player', 'Multi-player', 'MMO', 'Co-o...","Dec 6, 2024",2999.0
1,1,353,353,Palworld,1623730,incredibly fun and great replayability i think...,79,92.016667,2026-02-28 14:33:41,"50,000,000 .. 100,000,000",['Pocketpair'],['Pocketpair'],"['Action', 'Adventure', 'Indie', 'RPG', 'Early...",['windows'],"['Single-player', 'Multi-player', 'Co-op', 'On...","Jan 18, 2024",2999.0
2,2,1333,1333,Monster Hunter Wilds,2246340,i monster my hunter till i wild,32,113.883333,2026-03-01 10:51:44,"20,000,000 .. 50,000,000","['CAPCOM Co., Ltd.']","['CAPCOM Co., Ltd.']","['Action', 'Adventure', 'RPG']",['windows'],"['Single-player', 'Multi-player', 'Co-op', 'On...","Feb 27, 2025",6999.0
3,3,905,905,Left 4 Dead 2,550,this is a really good game if you love zombie ...,74,33.550000,2026-03-01 23:18:17,"50,000,000 .. 100,000,000",['Valve'],['Valve'],['Action'],"['windows', 'linux']","['Single-player', 'Multi-player', 'PvP', 'Onli...","Nov 16, 2009",999.0
4,4,1289,1289,War Thunder,236390,love the game but it piss me the off unlike an...,65,224.633333,2024-01-15 07:27:11,"20,000,000 .. 50,000,000",['Gaijin Entertainment'],['Gaijin Network Ltd'],"['Action', 'Massively Multiplayer', 'Simulatio...","['windows', 'mac', 'linux']","['Single-player', 'Multi-player', 'MMO', 'PvP'...","Aug 15, 2013",NaN


In [13]:
df_schema3 = run_pipeline(
    input_file       = r"C:\Users\King\Downloads\SCHEMA_2.csv",
    output_file      = "SCHEMA_3.csv",
    fix_spelling     = True,
    remove_stopwords = True,
)
df_schema3.head()


  INPUT : C:\Users\King\Downloads\SCHEMA_2.csv
  OUTPUT: SCHEMA_3.csv
Loaded 200 rows.

Fixing spelling...
Removing stopwords...

--- Summary ---
Rows             : 200
Empty text rows  : 6

Saved -> SCHEMA_3.csv


,Unnamed: 0.2,Unnamed: 0.1,Unnamed: 0,game_name,app_id,review_text,review_length,hours_played,review_date,owners,developers,publishers,genres,platforms,categories,release_date,price
0,0,1860,1860,Path of Exile 2,2694490,NaN,1,10.600000,2026-02-26 23:50:27,"20,000,000 .. 50,000,000",['Grinding Gear Games'],['Grinding Gear Games'],"['Action', 'Adventure', 'Massively Multiplayer...",['windows'],"['Single-player', 'Multi-player', 'MMO', 'Co-o...","Dec 6, 2024",2999.0
1,1,353,353,Palworld,1623730,incredibly fun great repeatability think spell...,79,92.016667,2026-02-28 14:33:41,"50,000,000 .. 100,000,000",['Pocketpair'],['Pocketpair'],"['Action', 'Adventure', 'Indie', 'RPG', 'Early...",['windows'],"['Single-player', 'Multi-player', 'Co-op', 'On...","Jan 18, 2024",2999.0
2,2,1333,1333,Monster Hunter Wilds,2246340,monster hunter till wild,32,113.883333,2026-03-01 10:51:44,"20,000,000 .. 50,000,000","['CAPCOM Co., Ltd.']","['CAPCOM Co., Ltd.']","['Action', 'Adventure', 'RPG']",['windows'],"['Single-player', 'Multi-player', 'Co-op', 'On...","Feb 27, 2025",6999.0
3,3,905,905,Left 4 Dead 2,550,really good game love zombie apocalypse game g...,74,33.550000,2026-03-01 23:18:17,"50,000,000 .. 100,000,000",['Valve'],['Valve'],['Action'],"['windows', 'linux']","['Single-player', 'Multi-player', 'PvP', 'Onli...","Nov 16, 2009",999.0
4,4,1289,1289,War Thunder,236390,love game piss unlike game,65,224.633333,2024-01-15 07:27:11,"20,000,000 .. 50,000,000",['Gaijin Entertainment'],['Gaijin Network Ltd'],"['Action', 'Massively Multiplayer', 'Simulatio...","['windows', 'mac', 'linux']","['Single-player', 'Multi-player', 'MMO', 'PvP'...","Aug 15, 2013",NaN


# SentiWordnet

In [4]:
sample_df=pd.read_csv(r"D:\final_ground_truth.CSV")
sample_df

,Unnamed: 0.1,Unnamed: 0,game_name,app_id,review_text,review_length,hours_played,review_date,owners,developers,publishers,genres,platforms,categories,release_date,price,label_1,label_2,label_3,final_label
0,1860,1860,Path of Exile 2,2694490,`,1,10.600000,2026-02-26 23:50:27,"20,000,000 .. 50,000,000",['Grinding Gear Games'],['Grinding Gear Games'],"['Action', 'Adventure', 'Massively Multiplayer...",['windows'],"['Single-player', 'Multi-player', 'MMO', 'Co-o...","Dec 6, 2024",2999.0,neutral,neutral,neutral,neutral
1,353,353,Palworld,1623730,incredibly fun and great replayability (i thin...,79,92.016667,2026-02-28 14:33:41,"50,000,000 .. 100,000,000",['Pocketpair'],['Pocketpair'],"['Action', 'Adventure', 'Indie', 'RPG', 'Early...",['windows'],"['Single-player', 'Multi-player', 'Co-op', 'On...","Jan 18, 2024",2999.0,positive,positive,positive,positive
2,1333,1333,Monster Hunter Wilds,2246340,I Monster my Hunter till I Wilds,32,113.883333,2026-03-01 10:51:44,"20,000,000 .. 50,000,000","['CAPCOM Co., Ltd.']","['CAPCOM Co., Ltd.']","['Action', 'Adventure', 'RPG']",['windows'],"['Single-player', 'Multi-player', 'Co-op', 'On...","Feb 27, 2025",6999.0,neutral,neutral,negative,neutral
3,905,905,Left 4 Dead 2,550,This is a really good game if you love Zombie ...,74,33.550000,2026-03-01 23:18:17,"50,000,000 .. 100,000,000",['Valve'],['Valve'],['Action'],"['windows', 'linux']","['Single-player', 'Multi-player', 'PvP', 'Onli...","Nov 16, 2009",999.0,positive,positive,positive,positive
4,1289,1289,War Thunder,236390,love the game but it pisses me the ♥♥♥♥ off un...,65,224.633333,2024-01-15 07:27:11,"20,000,000 .. 50,000,000",['Gaijin Entertainment'],['Gaijin Network Ltd'],"['Action', 'Massively Multiplayer', 'Simulatio...","['windows', 'mac', 'linux']","['Single-player', 'Multi-player', 'MMO', 'PvP'...","Aug 15, 2013",NaN,positive,negative,negative,negative
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
195,462,462,Team Fortress 2,440,the OG. The big chunga. This was one of the bi...,94,103.883333,2026-02-20 16:00:58,"50,000,000 .. 100,000,000",['Valve'],['Valve'],"['Action', 'Free To Play']","['windows', 'linux']","['Multi-player', 'Cross-Platform Multiplayer',...","Oct 10, 2007",NaN,positive,positive,positive,positive
196,1105,1105,Lost Ark,1599340,I'm so grateful I quit playing in June of 2022...,319,924.116667,2026-02-22 04:17:29,"50,000,000 .. 100,000,000",['Smilegate RPG'],['Amazon Game Studios'],"['Action', 'Adventure', 'Massively Multiplayer...",['windows'],"['Single-player', 'Multi-player', 'MMO', 'PvP'...","Feb 11, 2022",NaN,neutral,negative,negative,negative
197,855,855,Grand Theft Auto V Legacy,271590,best game,9,31.683333,2026-03-01 07:10:12,"50,000,000 .. 100,000,000",['Rockstar North'],['Rockstar Games'],"['Action', 'Adventure']",['windows'],"['Single-player', 'Multi-player', 'PvP', 'Onli...","Apr 13, 2015",NaN,positive,positive,positive,positive
198,693,693,New World: Aeternum,1063730,I started a few times over since the story cha...,592,1342.783333,2026-02-08 09:09:30,"50,000,000 .. 100,000,000",['Amazon Game Studios'],['Amazon Game Studios'],"['Action', 'Adventure', 'Massively Multiplayer...",['windows'],"['Multi-player', 'MMO', 'PvP', 'Online PvP', '...","Sep 28, 2021",NaN,negative,negative,negative,negative


In [5]:
import nltk
from nltk.corpus import sentiwordnet as swn
from nltk.corpus import wordnet as wn
from nltk.tokenize import word_tokenize
from nltk.tag import pos_tag
from nltk.stem import WordNetLemmatizer
import pandas as pd
import numpy as np

nltk.download('sentiwordnet')
nltk.download('wordnet')
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')




[nltk_data] Error loading sentiwordnet: <urlopen error [Errno 11001]
[nltk_data]     getaddrinfo failed>
[nltk_data] Error loading wordnet: <urlopen error [Errno 11001]
[nltk_data]     getaddrinfo failed>
[nltk_data] Error loading punkt: <urlopen error [Errno 11001]
[nltk_data]     getaddrinfo failed>
[nltk_data] Error loading averaged_perceptron_tagger: <urlopen error
[nltk_data]     [Errno 11001] getaddrinfo failed>


False

In [6]:
def get_wordnet_pos(treebank_tag):
    if treebank_tag.startswith('J'):
        return wn.ADJ
    elif treebank_tag.startswith('V'):
        return wn.VERB
    elif treebank_tag.startswith('N'):
        return wn.NOUN
    elif treebank_tag.startswith('R'):
        return wn.ADV
    else:
        return None
    


    

In [7]:
def get_sentiment_score(word, pos):
    lemmatizer = WordNetLemmatizer()
    lemma = lemmatizer.lemmatize(word, pos=pos)

    synsets = list(swn.senti_synsets(lemma, pos))

    if not synsets:
        return 0.0, 0.0, 1.0

    pos_score = np.mean([s.pos_score() for s in synsets])
    neg_score = np.mean([s.neg_score() for s in synsets])
    obj_score = np.mean([s.obj_score() for s in synsets])

    return pos_score, neg_score, obj_score

In [8]:
NEGATION_WORDS = {
    "not","no","never","neither","nor","nobody",
    "nothing","nowhere","hardly","scarcely","barely",
    "n't","nt","without","cannot","can't","won't",
    "isn't","aren't","wasn't","weren't","doesn't",
    "don't","didn't","hasn't","haven't","hadn't"
}

def apply_negation(tokens):
    WINDOW = 3
    negated_flags = [False] * len(tokens)
    neg_countdown = 0

    for i, token in enumerate(tokens):
        if token.lower() in NEGATION_WORDS:
            neg_countdown = WINDOW
        elif neg_countdown > 0:
            negated_flags[i] = True
            neg_countdown -= 1

    return list(zip(tokens, negated_flags))

In [9]:
def sentiwordnet_classify(text, threshold=0.05):
    tokens = word_tokenize(text.lower())
    tagged = pos_tag(tokens)
    token_neg_pairs = apply_negation(tokens)

    total_pos, total_neg = 0.0, 0.0
    scored_words = 0

    for (word, treebank_pos), (_, is_negated) in zip(tagged, token_neg_pairs):
        wn_pos = get_wordnet_pos(treebank_pos)

        if wn_pos is None:
            continue

        p, n, _ = get_sentiment_score(word, wn_pos)

        if p == 0 and n == 0:
            continue

        if is_negated:
            p, n = n, p

        total_pos += p
        total_neg += n
        scored_words += 1

    if scored_words == 0:
        return {"label":"neutral","net_score":0.0,
                "pos_score":0.0,"neg_score":0.0}

    avg_pos = total_pos / scored_words
    avg_neg = total_neg / scored_words
    net = avg_pos - avg_neg

    if net > threshold:
        label = "positive"
    elif net < -threshold:
        label = "negative"
    else:
        label = "neutral"

    return {
        "label": label,
        "net_score": round(net,4),
        "pos_score": round(avg_pos,4),
        "neg_score": round(avg_neg,4)
    }

In [10]:
data

,Unnamed: 0,game_name,app_id,review_text,review_length,hours_played,review_date,owners,developers,publishers,genres,platforms,categories,release_date,price
0,0,Counter-Strike: Global Offensive,730,pucajjj bam bam,15,197.216667,2026-03-02 01:55:35,"100,000,000 .. 200,000,000",['Valve'],['Valve'],"['Action', 'Free To Play']","['windows', 'linux']","['Multi-player', 'Cross-Platform Multiplayer',...","Aug 21, 2012",NaN
1,1,Counter-Strike: Global Offensive,730,YES,3,22.850000,2026-03-02 01:39:35,"100,000,000 .. 200,000,000",['Valve'],['Valve'],"['Action', 'Free To Play']","['windows', 'linux']","['Multi-player', 'Cross-Platform Multiplayer',...","Aug 21, 2012",NaN
2,2,Counter-Strike: Global Offensive,730,its pretty fun,14,24.150000,2026-03-02 01:32:54,"100,000,000 .. 200,000,000",['Valve'],['Valve'],"['Action', 'Free To Play']","['windows', 'linux']","['Multi-player', 'Cross-Platform Multiplayer',...","Aug 21, 2012",NaN
3,3,Counter-Strike: Global Offensive,730,awsone,6,264.666667,2026-03-02 01:30:30,"100,000,000 .. 200,000,000",['Valve'],['Valve'],"['Action', 'Free To Play']","['windows', 'linux']","['Multi-player', 'Cross-Platform Multiplayer',...","Aug 21, 2012",NaN
4,4,Counter-Strike: Global Offensive,730,s,1,136.316667,2026-03-02 01:30:21,"100,000,000 .. 200,000,000",['Valve'],['Valve'],"['Action', 'Free To Play']","['windows', 'linux']","['Multi-player', 'Cross-Platform Multiplayer',...","Aug 21, 2012",NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1995,1995,Wallpaper Engine,431960,Simply goated,13,3.616667,2026-02-28 21:36:51,"20,000,000 .. 50,000,000",['Wallpaper Engine Team'],['Wallpaper Engine Team'],"['Casual', 'Indie', 'Animation & Modeling', 'D...",['windows'],"['Steam Achievements', 'Steam Trading Cards', ...","Nov 16, 2018",499.0
1996,1996,Wallpaper Engine,431960,"""it cool""",9,0.183333,2026-02-28 21:30:32,"20,000,000 .. 50,000,000",['Wallpaper Engine Team'],['Wallpaper Engine Team'],"['Casual', 'Indie', 'Animation & Modeling', 'D...",['windows'],"['Steam Achievements', 'Steam Trading Cards', ...","Nov 16, 2018",499.0
1997,1997,Wallpaper Engine,431960,good\r\n,6,3.600000,2026-02-28 21:15:45,"20,000,000 .. 50,000,000",['Wallpaper Engine Team'],['Wallpaper Engine Team'],"['Casual', 'Indie', 'Animation & Modeling', 'D...",['windows'],"['Steam Achievements', 'Steam Trading Cards', ...","Nov 16, 2018",499.0
1998,1998,Wallpaper Engine,431960,i like cat,10,59.416667,2026-02-28 20:38:56,"20,000,000 .. 50,000,000",['Wallpaper Engine Team'],['Wallpaper Engine Team'],"['Casual', 'Indie', 'Animation & Modeling', 'D...",['windows'],"['Steam Achievements', 'Steam Trading Cards', ...","Nov 16, 2018",499.0


In [11]:
def classify_dataframe(data, text_column="review_text", threshold=0.05):
    results = data[text_column].apply(lambda t: sentiwordnet_classify(str(t), threshold))

    df = data.copy()
    df["swn_pos_score"] = results.apply(lambda r: r["pos_score"])
    df["swn_neg_score"] = results.apply(lambda r: r["neg_score"])
    df["swn_net_score"] = results.apply(lambda r: r["net_score"])
    df["swn_label"] = results.apply(lambda r: r["label"])

    return df

In [12]:
df_SentiWordnet = classify_dataframe(data)

df_SentiWordnet

,Unnamed: 0,game_name,app_id,review_text,review_length,hours_played,review_date,owners,developers,publishers,genres,platforms,categories,release_date,price,swn_pos_score,swn_neg_score,swn_net_score,swn_label
0,0,Counter-Strike: Global Offensive,730,pucajjj bam bam,15,197.216667,2026-03-02 01:55:35,"100,000,000 .. 200,000,000",['Valve'],['Valve'],"['Action', 'Free To Play']","['windows', 'linux']","['Multi-player', 'Cross-Platform Multiplayer',...","Aug 21, 2012",NaN,0.0000,0.1250,-0.1250,negative
1,1,Counter-Strike: Global Offensive,730,YES,3,22.850000,2026-03-02 01:39:35,"100,000,000 .. 200,000,000",['Valve'],['Valve'],"['Action', 'Free To Play']","['windows', 'linux']","['Multi-player', 'Cross-Platform Multiplayer',...","Aug 21, 2012",NaN,0.2500,0.0000,0.2500,positive
2,2,Counter-Strike: Global Offensive,730,its pretty fun,14,24.150000,2026-03-02 01:32:54,"100,000,000 .. 200,000,000",['Valve'],['Valve'],"['Action', 'Free To Play']","['windows', 'linux']","['Multi-player', 'Cross-Platform Multiplayer',...","Aug 21, 2012",NaN,0.1250,0.1562,-0.0312,neutral
3,3,Counter-Strike: Global Offensive,730,awsone,6,264.666667,2026-03-02 01:30:30,"100,000,000 .. 200,000,000",['Valve'],['Valve'],"['Action', 'Free To Play']","['windows', 'linux']","['Multi-player', 'Cross-Platform Multiplayer',...","Aug 21, 2012",NaN,0.0000,0.0000,0.0000,neutral
4,4,Counter-Strike: Global Offensive,730,s,1,136.316667,2026-03-02 01:30:21,"100,000,000 .. 200,000,000",['Valve'],['Valve'],"['Action', 'Free To Play']","['windows', 'linux']","['Multi-player', 'Cross-Platform Multiplayer',...","Aug 21, 2012",NaN,0.0000,0.0000,0.0000,neutral
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1995,1995,Wallpaper Engine,431960,Simply goated,13,3.616667,2026-02-28 21:36:51,"20,000,000 .. 50,000,000",['Wallpaper Engine Team'],['Wallpaper Engine Team'],"['Casual', 'Indie', 'Animation & Modeling', 'D...",['windows'],"['Steam Achievements', 'Steam Trading Cards', ...","Nov 16, 2018",499.0,0.3438,0.0938,0.2500,positive
1996,1996,Wallpaper Engine,431960,"""it cool""",9,0.183333,2026-02-28 21:30:32,"20,000,000 .. 50,000,000",['Wallpaper Engine Team'],['Wallpaper Engine Team'],"['Casual', 'Indie', 'Animation & Modeling', 'D...",['windows'],"['Steam Achievements', 'Steam Trading Cards', ...","Nov 16, 2018",499.0,0.2917,0.1458,0.1458,positive
1997,1997,Wallpaper Engine,431960,good\r\n,6,3.600000,2026-02-28 21:15:45,"20,000,000 .. 50,000,000",['Wallpaper Engine Team'],['Wallpaper Engine Team'],"['Casual', 'Indie', 'Animation & Modeling', 'D...",['windows'],"['Steam Achievements', 'Steam Trading Cards', ...","Nov 16, 2018",499.0,0.6190,0.0060,0.6131,positive
1998,1998,Wallpaper Engine,431960,i like cat,10,59.416667,2026-02-28 20:38:56,"20,000,000 .. 50,000,000",['Wallpaper Engine Team'],['Wallpaper Engine Team'],"['Casual', 'Indie', 'Animation & Modeling', 'D...",['windows'],"['Steam Achievements', 'Steam Trading Cards', ...","Nov 16, 2018",499.0,0.2000,0.0078,0.1922,positive


In [13]:
sample_df_SentiWordnet = classify_dataframe(sample_df,threshold=0.0)

sample_df_SentiWordnet

,Unnamed: 0.1,Unnamed: 0,game_name,app_id,review_text,review_length,hours_played,review_date,owners,developers,...,release_date,price,label_1,label_2,label_3,final_label,swn_pos_score,swn_neg_score,swn_net_score,swn_label
0,1860,1860,Path of Exile 2,2694490,`,1,10.600000,2026-02-26 23:50:27,"20,000,000 .. 50,000,000",['Grinding Gear Games'],...,"Dec 6, 2024",2999.0,neutral,neutral,neutral,neutral,0.0000,0.0000,0.0000,neutral
1,353,353,Palworld,1623730,incredibly fun and great replayability (i thin...,79,92.016667,2026-02-28 14:33:41,"50,000,000 .. 100,000,000",['Pocketpair'],...,"Jan 18, 2024",2999.0,positive,positive,positive,positive,0.1413,0.1373,0.0040,positive
2,1333,1333,Monster Hunter Wilds,2246340,I Monster my Hunter till I Wilds,32,113.883333,2026-03-01 10:51:44,"20,000,000 .. 50,000,000","['CAPCOM Co., Ltd.']",...,"Feb 27, 2025",6999.0,neutral,neutral,negative,neutral,0.0000,0.1229,-0.1229,negative
3,905,905,Left 4 Dead 2,550,This is a really good game if you love Zombie ...,74,33.550000,2026-03-01 23:18:17,"50,000,000 .. 100,000,000",['Valve'],...,"Nov 16, 2009",999.0,positive,positive,positive,positive,0.3125,0.0322,0.2803,positive
4,1289,1289,War Thunder,236390,love the game but it pisses me the ♥♥♥♥ off un...,65,224.633333,2024-01-15 07:27:11,"20,000,000 .. 50,000,000",['Gaijin Entertainment'],...,"Aug 15, 2013",NaN,positive,negative,negative,negative,0.1911,0.1087,0.0824,positive
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
195,462,462,Team Fortress 2,440,the OG. The big chunga. This was one of the bi...,94,103.883333,2026-02-20 16:00:58,"50,000,000 .. 100,000,000",['Valve'],...,"Oct 10, 2007",NaN,positive,positive,positive,positive,0.1030,0.0945,0.0085,positive
196,1105,1105,Lost Ark,1599340,I'm so grateful I quit playing in June of 2022...,319,924.116667,2026-02-22 04:17:29,"50,000,000 .. 100,000,000",['Smilegate RPG'],...,"Feb 11, 2022",NaN,neutral,negative,negative,negative,0.1719,0.0821,0.0898,positive
197,855,855,Grand Theft Auto V Legacy,271590,best game,9,31.683333,2026-03-01 07:10:12,"50,000,000 .. 100,000,000",['Rockstar North'],...,"Apr 13, 2015",NaN,positive,positive,positive,positive,0.3266,0.0254,0.3011,positive
198,693,693,New World: Aeternum,1063730,I started a few times over since the story cha...,592,1342.783333,2026-02-08 09:09:30,"50,000,000 .. 100,000,000",['Amazon Game Studios'],...,"Sep 28, 2021",NaN,negative,negative,negative,negative,0.0760,0.0674,0.0086,positive


In [14]:
from sklearn.metrics import classification_report

print("SentiWordNet")
print(classification_report(sample_df_SentiWordnet["final_label"], sample_df_SentiWordnet["swn_label"]))

SentiWordNet
              precision    recall  f1-score   support

    negative       0.37      0.43      0.40        49
     neutral       0.62      0.66      0.64        35
    positive       0.74      0.67      0.70       116

    accuracy                           0.61       200
   macro avg       0.58      0.59      0.58       200
weighted avg       0.63      0.61      0.62       200



In [15]:
from sklearn.metrics import accuracy_score, f1_score

y_true = sample_df_SentiWordnet["final_label"]
y_pred = sample_df_SentiWordnet["swn_label"]

acc_sent = accuracy_score(y_true, y_pred)
f1_sent = f1_score(y_true, y_pred, average="weighted")  # مهم للـ multi-class أو imbalance

print("Accuracy:", acc_sent)
print("F1 Score:", f1_sent)

Accuracy: 0.61
F1 Score: 0.6164485948212364


# Bing Liu Opinion Lexicon

In [16]:
import re
import pandas as pd
from collections import Counter
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, f1_score
import matplotlib.pyplot as plt


In [17]:
POSITIVE_WORDS = set("""
abound abundance abundant accurate achievable admirable admire adorable
adore adroit affable agile alert altruistic amaze amazing amicable ample
amusing appreciate appreciated aspiring astounding authentic balanced
beauteous beautiful beauty beneficial best blessed bliss blissful brave
breathtaking bright brilliant calm capable caring celebrated champion
cheerful clean clever comfortable commendable compassionate competent
confident cool cooperative correct courageous crisp cute decent dedicated
dependable deserve devoted diligent distinct dynamic effective efficient
elegant empathetic encouraging enjoyable enthusiastic exceptional excellent
exciting exhilarating exquisite faithful fantastic fast favorable fine
flexible friendly fun generous genuine good graceful great happy helpful
honorable impressive incredible ingenious innovative inspiring intelligent
invincible joyful kind laudable lively logical lovable lovely loyal
magnificent masterful meritorious mindful natural nice noble outstanding
patient peaceful perfect persevering phenomenal pleasant positive powerful
productive proficient progressive quick reliable remarkable resilient
resourceful satisfying smart solid special spectacular stunning successful
sweet talented thankful thorough trustworthy useful valuable vibrant warm
wonderful worthy addictive engrossing gripping compelling polished smooth
immersive responsive tight flawless crisp rewarding delightful charming
witty clever engaging intuitive accessible deep rich varied enjoyable
superb tremendous terrific stellar flawless masterpiece unique refreshing
underrated gem hidden treasure must-play outstanding smooth fluid dynamic
""".split())

NEGATIVE_WORDS = set("""
abrupt absurd awful bad broken buggy cheap clunky confusing corrupt crash
crashes defective delayed disappointing disgusting dull error fail failure
fake faulty frustrating garbage glitch horrible inconsistent inferior irritating
lag lagging messy mediocre nasty never poor problem repetitive ridiculous
rude slow sluggish terrible toxic unreliable useless waste worthless abysmal
annoying atrocious awful bland boring broken buggy chaotic choppy clumsy
crashing dreadful empty excessive exploitative fake fragile frustrating grindy
hollow janky limited mediocre monotonous outdated overpriced painful paywall
pointless punishing shallow sloppy terrible trash underwhelming unfair
unfinished unpolished unresponsive unstable waste broken empty hollow cheating
lying manipulative predatory scam misleading repetitive tedious boring dull
unbalanced unfair unfulfilling unsatisfying lackluster forgettable generic
copy-paste clone bland disappointing letdown broken promises overhyped
""".split())

NEGATION_WORDS = {
    "not", "no", "never", "neither", "nor", "hardly", "barely",
    "scarcely", "doesn't", "don't", "didn't", "isn't", "wasn't",
    "aren't", "weren't", "won't", "wouldn't", "can't", "cannot",
    "couldn't", "shouldn't", "nothing", "nobody", "nowhere"
}

INTENSIFIERS = {
    "very": 1.5, "extremely": 2.0, "absolutely": 2.0, "totally": 1.5,
    "completely": 1.8, "really": 1.5, "super": 1.5, "incredibly": 2.0,
    "truly": 1.5, "insanely": 2.0,"ridiculously": 1.8, "so": 1.3, "quite": 1.2,
    "pretty": 1.2, "highly": 1.5, "deeply": 1.5,"genuinely": 1.3,
    "especially": 1.3
}


### 2. Preprocessing & Scoring Functions

In [18]:
def preprocess(text):
    if pd.isna(text) or text == "":
        return []
    text = re.sub(r"[^a-z\s']", " ", str(text).lower())
    return text.split()

def score_review(text):
    tokens = preprocess(text)
    if not tokens:
        return {"raw_score": 0, "pos_count": 0, "neg_count": 0,
                "pos_words_found": [], "neg_words_found": []}

    raw_score, pos_count, neg_count = 0.0, 0, 0
    pos_found, neg_found = [], []
    negation_counter, intensifier_mult = 0, 1.0

    for token in tokens:
        if token in NEGATION_WORDS:
            negation_counter = 3
            intensifier_mult = 1.0
            continue
        if token in INTENSIFIERS:
            intensifier_mult = INTENSIFIERS[token]
            if negation_counter > 0: negation_counter -= 1
            continue

        word_score = 0.0
        if token in POSITIVE_WORDS:
            word_score = +1.0; pos_count += 1; pos_found.append(token)
        elif token in NEGATIVE_WORDS:
            word_score = -1.0; neg_count += 1; neg_found.append(token)

        if word_score != 0:
            word_score *= intensifier_mult
            intensifier_mult = 1.0
            if negation_counter > 0:
                word_score *= -1
                negation_counter -= 1
        else:
            if negation_counter > 0: negation_counter -= 1
            intensifier_mult = 1.0

        raw_score += word_score

    return {"raw_score": raw_score, "pos_count": pos_count, "neg_count": neg_count,
            "pos_words_found": pos_found, "neg_words_found": neg_found}

def classify(score, threshold=0.5):
    if score > threshold:  return "positive"
    if score < -threshold: return "negative"
    return "neutral"


In [19]:
df_Bing_Liu=data.copy()
results = df_Bing_Liu["review_text"].apply(score_review)
df_Bing_Liu["raw_score"]= results.apply(lambda r: r["raw_score"])
df_Bing_Liu["pos_hits"]= results.apply(lambda r: r["pos_count"])
df_Bing_Liu["neg_hits"]= results.apply(lambda r: r["neg_count"])
df_Bing_Liu["pos_words_found"] = results.apply(lambda r: ", ".join(r["pos_words_found"]))
df_Bing_Liu["neg_words_found"] = results.apply(lambda r: ", ".join(r["neg_words_found"]))
df_Bing_Liu["predicted_label"] = df_Bing_Liu["raw_score"].apply(classify)
df_Bing_Liu

,Unnamed: 0,game_name,app_id,review_text,review_length,hours_played,review_date,owners,developers,publishers,...,platforms,categories,release_date,price,raw_score,pos_hits,neg_hits,pos_words_found,neg_words_found,predicted_label
0,0,Counter-Strike: Global Offensive,730,pucajjj bam bam,15,197.216667,2026-03-02 01:55:35,"100,000,000 .. 200,000,000",['Valve'],['Valve'],...,"['windows', 'linux']","['Multi-player', 'Cross-Platform Multiplayer',...","Aug 21, 2012",NaN,0.0,0,0,,,neutral
1,1,Counter-Strike: Global Offensive,730,YES,3,22.850000,2026-03-02 01:39:35,"100,000,000 .. 200,000,000",['Valve'],['Valve'],...,"['windows', 'linux']","['Multi-player', 'Cross-Platform Multiplayer',...","Aug 21, 2012",NaN,0.0,0,0,,,neutral
2,2,Counter-Strike: Global Offensive,730,its pretty fun,14,24.150000,2026-03-02 01:32:54,"100,000,000 .. 200,000,000",['Valve'],['Valve'],...,"['windows', 'linux']","['Multi-player', 'Cross-Platform Multiplayer',...","Aug 21, 2012",NaN,1.2,1,0,fun,,positive
3,3,Counter-Strike: Global Offensive,730,awsone,6,264.666667,2026-03-02 01:30:30,"100,000,000 .. 200,000,000",['Valve'],['Valve'],...,"['windows', 'linux']","['Multi-player', 'Cross-Platform Multiplayer',...","Aug 21, 2012",NaN,0.0,0,0,,,neutral
4,4,Counter-Strike: Global Offensive,730,s,1,136.316667,2026-03-02 01:30:21,"100,000,000 .. 200,000,000",['Valve'],['Valve'],...,"['windows', 'linux']","['Multi-player', 'Cross-Platform Multiplayer',...","Aug 21, 2012",NaN,0.0,0,0,,,neutral
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1995,1995,Wallpaper Engine,431960,Simply goated,13,3.616667,2026-02-28 21:36:51,"20,000,000 .. 50,000,000",['Wallpaper Engine Team'],['Wallpaper Engine Team'],...,['windows'],"['Steam Achievements', 'Steam Trading Cards', ...","Nov 16, 2018",499.0,0.0,0,0,,,neutral
1996,1996,Wallpaper Engine,431960,"""it cool""",9,0.183333,2026-02-28 21:30:32,"20,000,000 .. 50,000,000",['Wallpaper Engine Team'],['Wallpaper Engine Team'],...,['windows'],"['Steam Achievements', 'Steam Trading Cards', ...","Nov 16, 2018",499.0,1.0,1,0,cool,,positive
1997,1997,Wallpaper Engine,431960,good\r\n,6,3.600000,2026-02-28 21:15:45,"20,000,000 .. 50,000,000",['Wallpaper Engine Team'],['Wallpaper Engine Team'],...,['windows'],"['Steam Achievements', 'Steam Trading Cards', ...","Nov 16, 2018",499.0,1.0,1,0,good,,positive
1998,1998,Wallpaper Engine,431960,i like cat,10,59.416667,2026-02-28 20:38:56,"20,000,000 .. 50,000,000",['Wallpaper Engine Team'],['Wallpaper Engine Team'],...,['windows'],"['Steam Achievements', 'Steam Trading Cards', ...","Nov 16, 2018",499.0,0.0,0,0,,,neutral


In [20]:
sample_df_Bing_Liu=sample_df.copy()
results = sample_df_Bing_Liu["review_text"].apply(score_review)
sample_df_Bing_Liu["raw_score"]= results.apply(lambda r: r["raw_score"])
sample_df_Bing_Liu["pos_hits"]= results.apply(lambda r: r["pos_count"])
sample_df_Bing_Liu["neg_hits"]= results.apply(lambda r: r["neg_count"])
sample_df_Bing_Liu["pos_words_found"] = results.apply(lambda r: ", ".join(r["pos_words_found"]))
sample_df_Bing_Liu["neg_words_found"] = results.apply(lambda r: ", ".join(r["neg_words_found"]))
sample_df_Bing_Liu["predicted_label"] = sample_df_Bing_Liu["raw_score"].apply(classify)
sample_df_Bing_Liu

,Unnamed: 0.1,Unnamed: 0,game_name,app_id,review_text,review_length,hours_played,review_date,owners,developers,...,label_1,label_2,label_3,final_label,raw_score,pos_hits,neg_hits,pos_words_found,neg_words_found,predicted_label
0,1860,1860,Path of Exile 2,2694490,`,1,10.600000,2026-02-26 23:50:27,"20,000,000 .. 50,000,000",['Grinding Gear Games'],...,neutral,neutral,neutral,neutral,0.0,0,0,,,neutral
1,353,353,Palworld,1623730,incredibly fun and great replayability (i thin...,79,92.016667,2026-02-28 14:33:41,"50,000,000 .. 100,000,000",['Pocketpair'],...,positive,positive,positive,positive,3.0,2,0,"fun, great",,positive
2,1333,1333,Monster Hunter Wilds,2246340,I Monster my Hunter till I Wilds,32,113.883333,2026-03-01 10:51:44,"20,000,000 .. 50,000,000","['CAPCOM Co., Ltd.']",...,neutral,neutral,negative,neutral,0.0,0,0,,,neutral
3,905,905,Left 4 Dead 2,550,This is a really good game if you love Zombie ...,74,33.550000,2026-03-01 23:18:17,"50,000,000 .. 100,000,000",['Valve'],...,positive,positive,positive,positive,2.5,2,0,"good, good",,positive
4,1289,1289,War Thunder,236390,love the game but it pisses me the ♥♥♥♥ off un...,65,224.633333,2024-01-15 07:27:11,"20,000,000 .. 50,000,000",['Gaijin Entertainment'],...,positive,negative,negative,negative,0.0,0,0,,,neutral
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
195,462,462,Team Fortress 2,440,the OG. The big chunga. This was one of the bi...,94,103.883333,2026-02-20 16:00:58,"50,000,000 .. 100,000,000",['Valve'],...,positive,positive,positive,positive,0.0,0,0,,,neutral
196,1105,1105,Lost Ark,1599340,I'm so grateful I quit playing in June of 2022...,319,924.116667,2026-02-22 04:17:29,"50,000,000 .. 100,000,000",['Smilegate RPG'],...,neutral,negative,negative,negative,4.5,4,0,"fun, great, wonderful, good",,positive
197,855,855,Grand Theft Auto V Legacy,271590,best game,9,31.683333,2026-03-01 07:10:12,"50,000,000 .. 100,000,000",['Rockstar North'],...,positive,positive,positive,positive,1.0,1,0,best,,positive
198,693,693,New World: Aeternum,1063730,I started a few times over since the story cha...,592,1342.783333,2026-02-08 09:09:30,"50,000,000 .. 100,000,000",['Amazon Game Studios'],...,negative,negative,negative,negative,2.0,2,0,"correct, accessible",,positive


In [21]:
from sklearn.metrics import classification_report, accuracy_score, f1_score

y_true = sample_df_Bing_Liu["final_label"]          # ground truth
y_pred = sample_df_Bing_Liu["predicted_label"]      # model prediction

print("=== Bing Liu Evaluation ===\n")

# 1) Classification Report
print(classification_report(y_true, y_pred))



=== Bing Liu Evaluation ===

              precision    recall  f1-score   support

    negative       0.82      0.18      0.30        49
     neutral       0.35      0.97      0.52        35
    positive       0.83      0.66      0.74       116

    accuracy                           0.60       200
   macro avg       0.67      0.61      0.52       200
weighted avg       0.74      0.60      0.59       200



In [22]:
# 2) Accuracy + F1 Score
acc_bing = accuracy_score(y_true, y_pred)
f1_bing = f1_score(y_true, y_pred, average="weighted")

print("\nAccuracy:", acc_bing)
print("F1 Score (weighted):", f1_bing)


Accuracy: 0.6
F1 Score (weighted): 0.5917081157091201


# Text Representation

### Bag Of Words by 3 schema

In [23]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer

In [24]:
data_A = pd.read_csv(r"C:\Users\King\Downloads\SCHEMA_1.csv")
data_B = pd.read_csv(r"C:\Users\King\Downloads\SCHEMA_2.csv")
data_C = pd.read_csv(r"C:\Users\King\Downloads\SCHEMA_3.csv")

data_A

,Unnamed: 0.2,Unnamed: 0.1,Unnamed: 0,game_name,app_id,review_text,review_length,hours_played,review_date,owners,developers,publishers,genres,platforms,categories,release_date,price
0,0,1860,1860,Path of Exile 2,2694490,NaN,1,10.600000,2026-02-26 23:50:27,"20,000,000 .. 50,000,000",['Grinding Gear Games'],['Grinding Gear Games'],"['Action', 'Adventure', 'Massively Multiplayer...",['windows'],"['Single-player', 'Multi-player', 'MMO', 'Co-o...","Dec 6, 2024",2999.0
1,1,353,353,Palworld,1623730,incredibly fun and great replayability i thin...,79,92.016667,2026-02-28 14:33:41,"50,000,000 .. 100,000,000",['Pocketpair'],['Pocketpair'],"['Action', 'Adventure', 'Indie', 'RPG', 'Early...",['windows'],"['Single-player', 'Multi-player', 'Co-op', 'On...","Jan 18, 2024",2999.0
2,2,1333,1333,Monster Hunter Wilds,2246340,i monster my hunter till i wilds,32,113.883333,2026-03-01 10:51:44,"20,000,000 .. 50,000,000","['CAPCOM Co., Ltd.']","['CAPCOM Co., Ltd.']","['Action', 'Adventure', 'RPG']",['windows'],"['Single-player', 'Multi-player', 'Co-op', 'On...","Feb 27, 2025",6999.0
3,3,905,905,Left 4 Dead 2,550,this is a really good game if you love zombie ...,74,33.550000,2026-03-01 23:18:17,"50,000,000 .. 100,000,000",['Valve'],['Valve'],['Action'],"['windows', 'linux']","['Single-player', 'Multi-player', 'PvP', 'Onli...","Nov 16, 2009",999.0
4,4,1289,1289,War Thunder,236390,love the game but it pisses me the off unlike...,65,224.633333,2024-01-15 07:27:11,"20,000,000 .. 50,000,000",['Gaijin Entertainment'],['Gaijin Network Ltd'],"['Action', 'Massively Multiplayer', 'Simulatio...","['windows', 'mac', 'linux']","['Single-player', 'Multi-player', 'MMO', 'PvP'...","Aug 15, 2013",NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
195,195,462,462,Team Fortress 2,440,the og the big chunga this was one of the bi...,94,103.883333,2026-02-20 16:00:58,"50,000,000 .. 100,000,000",['Valve'],['Valve'],"['Action', 'Free To Play']","['windows', 'linux']","['Multi-player', 'Cross-Platform Multiplayer',...","Oct 10, 2007",NaN
196,196,1105,1105,Lost Ark,1599340,i m so grateful i quit playing in june of 2022...,319,924.116667,2026-02-22 04:17:29,"50,000,000 .. 100,000,000",['Smilegate RPG'],['Amazon Game Studios'],"['Action', 'Adventure', 'Massively Multiplayer...",['windows'],"['Single-player', 'Multi-player', 'MMO', 'PvP'...","Feb 11, 2022",NaN
197,197,855,855,Grand Theft Auto V Legacy,271590,best game,9,31.683333,2026-03-01 07:10:12,"50,000,000 .. 100,000,000",['Rockstar North'],['Rockstar Games'],"['Action', 'Adventure']",['windows'],"['Single-player', 'Multi-player', 'PvP', 'Onli...","Apr 13, 2015",NaN
198,198,693,693,New World: Aeternum,1063730,i started a few times over since the story cha...,592,1342.783333,2026-02-08 09:09:30,"50,000,000 .. 100,000,000",['Amazon Game Studios'],['Amazon Game Studios'],"['Action', 'Adventure', 'Massively Multiplayer...",['windows'],"['Multi-player', 'MMO', 'PvP', 'Online PvP', '...","Sep 28, 2021",NaN


In [25]:
TEXT_COLUMN = "review_text"   

def bow_transform(df, text_column, max_features=5000):

    texts = df[text_column].fillna("").astype(str)

    vectorizer = CountVectorizer(
        max_features=max_features,
        lowercase=False
    )

    X = vectorizer.fit_transform(texts)

    bow_df = pd.DataFrame(
        X.toarray(),
        columns=vectorizer.get_feature_names_out()
    )

    return bow_df

In [26]:
bow_A = bow_transform(data_A, TEXT_COLUMN)

bow_A.head()

,10,100,10000,11,15,18,19,20,2000s,2014,...,yet,you,your,youre,yuh,zdsrfg,zombie,zombies,zone,zones
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,1,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,1,0,0,0,0,1,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [27]:
bow_B = bow_transform(data_B, TEXT_COLUMN)

bow_B.head()

,aaa,ability,able,about,absolute,absolutely,absurd,acceptable,access,accessible,...,year,yes,yet,you,your,youre,yuh,zdsrfg,zombie,zone
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,1,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,1,0,0,0,0,1,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [28]:
bow_C = bow_transform(data_C, TEXT_COLUMN)

bow_C.head()

,ability,able,absolute,absolutely,absurd,acceptable,access,accessible,account,across,...,wow,writing,wrong,yeah,year,yes,yet,zdsrfg,zombie,zone
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [29]:
bow_A.to_csv("bow_A.csv", index=False)
bow_B.to_csv("bow_B.csv", index=False)
bow_C.to_csv("bow_C.csv", index=False)

### Glove

In [30]:
def load_glove(path):
    embeddings = {}
    
    with open(path, "r", encoding="utf8") as f:
        for line in f:
            values = line.split()
            word = values[0]
            vector = np.asarray(values[1:], dtype="float32")
            embeddings[word] = vector
            
    return embeddings


glove_path = "D:\chrom download\glove.6B\glove.6B.100d.txt"
glove = load_glove(glove_path)

print("Loaded words:", len(glove))

Loaded words: 400000


In [31]:
def sentence_to_glove(sentence, embeddings, dim=100):
    
    words = str(sentence).lower().split()
    vectors = []

    for w in words:
        if w in embeddings:
            vectors.append(embeddings[w])

    if len(vectors) == 0:
        return np.zeros(dim)

    return np.mean(vectors, axis=0)

In [32]:
def glove_transform(df, text_column, embeddings, dim=100):
    
    vectors = df[text_column].apply(
        lambda x: sentence_to_glove(x, embeddings, dim)
    )

    glove_df = pd.DataFrame(vectors.tolist())
    
    return glove_df

In [33]:
glove_A = glove_transform(data_A, TEXT_COLUMN, glove, dim=100)
glove_A.head()

,0,1,2,3,4,5,6,7,8,9,...,90,91,92,93,94,95,96,97,98,99
0,-0.040383,-0.213200,0.055937,-0.070654,0.465700,0.149880,0.350690,0.388610,1.038200,0.311420,...,0.915820,0.466170,0.100320,0.252650,-0.360120,-0.428180,-0.380400,-1.392700,-0.015867,0.113110
1,-0.143825,0.344769,0.484638,-0.299102,-0.441964,0.135739,-0.184393,0.019756,-0.076521,-0.123794,...,-0.095606,-0.008274,-0.013764,-0.004296,-0.282469,-0.152264,-0.320854,-0.395841,0.172938,0.483697
2,-0.059200,0.171948,0.506700,-0.411867,-0.510544,0.257052,0.106212,0.233714,-0.122674,-0.256370,...,-0.039907,-0.099103,0.386687,0.441575,-0.284024,-0.104528,-0.143750,-0.044171,0.028191,0.119464
3,-0.154745,0.328781,0.642082,-0.528432,-0.330222,0.355505,0.002228,-0.035931,-0.135312,-0.195115,...,-0.154298,-0.058036,-0.004499,0.089759,-0.324087,-0.189027,-0.165105,-0.295020,0.350665,0.419362
4,-0.096591,0.120905,0.653527,-0.439818,-0.178908,0.321480,-0.025387,0.033117,-0.041833,-0.177158,...,-0.216496,-0.111707,-0.179593,0.123989,-0.424695,-0.179251,-0.454657,-0.015595,0.423462,0.291503


In [34]:
glove_B = glove_transform(data_B, TEXT_COLUMN, glove, dim=100)
glove_B.head()

,0,1,2,3,4,5,6,7,8,9,...,90,91,92,93,94,95,96,97,98,99
0,-0.040383,-0.213200,0.055937,-0.070654,0.465700,0.149880,0.350690,0.388610,1.038200,0.311420,...,0.915820,0.466170,0.100320,0.252650,-0.360120,-0.428180,-0.380400,-1.392700,-0.015867,0.113110
1,-0.143825,0.344769,0.484638,-0.299102,-0.441964,0.135739,-0.184393,0.019756,-0.076521,-0.123794,...,-0.095606,-0.008274,-0.013764,-0.004296,-0.282469,-0.152264,-0.320854,-0.395841,0.172938,0.483697
2,-0.071858,0.159181,0.545474,-0.422754,-0.501642,0.282446,0.182295,0.202286,-0.177085,-0.407381,...,-0.101931,-0.167615,0.347618,0.357946,-0.442950,-0.130969,-0.329699,-0.045593,0.121638,0.105801
3,-0.161145,0.299406,0.689881,-0.524231,-0.370958,0.383974,0.045654,-0.047028,-0.117276,-0.204313,...,-0.153524,-0.081246,-0.060406,0.099712,-0.305121,-0.212704,-0.137307,-0.313678,0.354886,0.423804
4,-0.109095,0.150978,0.562983,-0.499467,-0.192168,0.361037,-0.071108,0.044518,-0.123832,-0.167358,...,-0.233470,-0.169667,-0.158649,0.150049,-0.463288,-0.262684,-0.424320,-0.055440,0.417841,0.234857


In [35]:
glove_C = glove_transform(data_C, TEXT_COLUMN, glove, dim=100)
glove_C.head()

,0,1,2,3,4,5,6,7,8,9,...,90,91,92,93,94,95,96,97,98,99
0,-0.040383,-0.213200,0.055937,-0.070654,0.465700,0.149880,0.350690,0.388610,1.038200,0.311420,...,0.915820,0.466170,0.100320,0.252650,-0.360120,-0.428180,-0.380400,-1.392700,-0.015867,0.113110
1,-0.257483,0.233369,0.394128,-0.176842,-0.469330,0.128710,-0.191124,-0.099982,-0.207650,-0.148254,...,-0.148027,0.108041,-0.007111,0.037179,-0.130632,-0.206946,-0.342146,-0.259017,-0.022690,0.498424
2,-0.122551,-0.004111,0.491177,-0.394060,-0.095974,0.111831,0.350261,0.090435,-0.369677,-0.691230,...,0.094160,-0.166844,0.145196,0.360324,-0.291268,-0.181020,-0.385767,0.195398,0.150452,-0.470122
3,0.050376,0.239954,0.710054,-0.597261,-0.459596,0.347771,0.150557,-0.373957,-0.217045,-0.282146,...,-0.110638,-0.011270,0.070122,0.177677,-0.302968,-0.430537,-0.162872,-0.182221,0.239884,0.354337
4,-0.052270,0.258955,0.498818,-0.545704,-0.268470,0.649430,0.196400,-0.156294,-0.389242,-0.323942,...,-0.290310,-0.094630,-0.095985,0.113128,-0.193524,-0.456304,-0.474642,0.042334,0.222160,0.103138


In [36]:
glove_A.to_csv("glove_A.csv", index=False)
glove_B.to_csv("glove_B.csv", index=False)
glove_C.to_csv("glove_C.csv", index=False)

# Machine Learning-Based Modelling 

### RANDOM FOREST 

In [37]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_selection import VarianceThreshold
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings("ignore")

In [38]:
sample_df

,Unnamed: 0.1,Unnamed: 0,game_name,app_id,review_text,review_length,hours_played,review_date,owners,developers,publishers,genres,platforms,categories,release_date,price,label_1,label_2,label_3,final_label
0,1860,1860,Path of Exile 2,2694490,`,1,10.600000,2026-02-26 23:50:27,"20,000,000 .. 50,000,000",['Grinding Gear Games'],['Grinding Gear Games'],"['Action', 'Adventure', 'Massively Multiplayer...",['windows'],"['Single-player', 'Multi-player', 'MMO', 'Co-o...","Dec 6, 2024",2999.0,neutral,neutral,neutral,neutral
1,353,353,Palworld,1623730,incredibly fun and great replayability (i thin...,79,92.016667,2026-02-28 14:33:41,"50,000,000 .. 100,000,000",['Pocketpair'],['Pocketpair'],"['Action', 'Adventure', 'Indie', 'RPG', 'Early...",['windows'],"['Single-player', 'Multi-player', 'Co-op', 'On...","Jan 18, 2024",2999.0,positive,positive,positive,positive
2,1333,1333,Monster Hunter Wilds,2246340,I Monster my Hunter till I Wilds,32,113.883333,2026-03-01 10:51:44,"20,000,000 .. 50,000,000","['CAPCOM Co., Ltd.']","['CAPCOM Co., Ltd.']","['Action', 'Adventure', 'RPG']",['windows'],"['Single-player', 'Multi-player', 'Co-op', 'On...","Feb 27, 2025",6999.0,neutral,neutral,negative,neutral
3,905,905,Left 4 Dead 2,550,This is a really good game if you love Zombie ...,74,33.550000,2026-03-01 23:18:17,"50,000,000 .. 100,000,000",['Valve'],['Valve'],['Action'],"['windows', 'linux']","['Single-player', 'Multi-player', 'PvP', 'Onli...","Nov 16, 2009",999.0,positive,positive,positive,positive
4,1289,1289,War Thunder,236390,love the game but it pisses me the ♥♥♥♥ off un...,65,224.633333,2024-01-15 07:27:11,"20,000,000 .. 50,000,000",['Gaijin Entertainment'],['Gaijin Network Ltd'],"['Action', 'Massively Multiplayer', 'Simulatio...","['windows', 'mac', 'linux']","['Single-player', 'Multi-player', 'MMO', 'PvP'...","Aug 15, 2013",NaN,positive,negative,negative,negative
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
195,462,462,Team Fortress 2,440,the OG. The big chunga. This was one of the bi...,94,103.883333,2026-02-20 16:00:58,"50,000,000 .. 100,000,000",['Valve'],['Valve'],"['Action', 'Free To Play']","['windows', 'linux']","['Multi-player', 'Cross-Platform Multiplayer',...","Oct 10, 2007",NaN,positive,positive,positive,positive
196,1105,1105,Lost Ark,1599340,I'm so grateful I quit playing in June of 2022...,319,924.116667,2026-02-22 04:17:29,"50,000,000 .. 100,000,000",['Smilegate RPG'],['Amazon Game Studios'],"['Action', 'Adventure', 'Massively Multiplayer...",['windows'],"['Single-player', 'Multi-player', 'MMO', 'PvP'...","Feb 11, 2022",NaN,neutral,negative,negative,negative
197,855,855,Grand Theft Auto V Legacy,271590,best game,9,31.683333,2026-03-01 07:10:12,"50,000,000 .. 100,000,000",['Rockstar North'],['Rockstar Games'],"['Action', 'Adventure']",['windows'],"['Single-player', 'Multi-player', 'PvP', 'Onli...","Apr 13, 2015",NaN,positive,positive,positive,positive
198,693,693,New World: Aeternum,1063730,I started a few times over since the story cha...,592,1342.783333,2026-02-08 09:09:30,"50,000,000 .. 100,000,000",['Amazon Game Studios'],['Amazon Game Studios'],"['Action', 'Adventure', 'Massively Multiplayer...",['windows'],"['Multi-player', 'MMO', 'PvP', 'Online PvP', '...","Sep 28, 2021",NaN,negative,negative,negative,negative


In [39]:
print("="*50)
print("LOADING DATA")
print("="*50)


df_scheme_a = pd.read_csv(r"D:\chrom download\bow_scheme_a.csv", index_col=0)
df_scheme_b = pd.read_csv(r"D:\chrom download\bow_scheme_b.csv", index_col=0)
df_scheme_c = pd.read_csv(r"D:\chrom download\bow_scheme_c.csv", index_col=0)

print(f"Scheme A: {df_scheme_a.shape}")
print(f"Scheme B: {df_scheme_b.shape}")
print(f"Scheme C: {df_scheme_c.shape}")

print("\n" + "="*50)
print("ALIGNING DATA")
print("="*50)

min_rows = min(len(df_scheme_a), len(df_scheme_b), len(df_scheme_c), len(sample_df))
print(f"Using {min_rows} rows")

df_scheme_a = df_scheme_a.iloc[:min_rows].copy()
df_scheme_b = df_scheme_b.iloc[:min_rows].copy()
df_scheme_c = df_scheme_c.iloc[:min_rows].copy()
df_truth_aligned = sample_df.iloc[:min_rows].copy()

df_scheme_a['final_label'] = df_truth_aligned['final_label'].values
df_scheme_b['final_label'] = df_truth_aligned['final_label'].values
df_scheme_c['final_label'] = df_truth_aligned['final_label'].values

print("\n" + "="*50)
print("LABEL ENCODING")
print("="*50)

encoder = LabelEncoder()

df_scheme_a['label_encoded'] = encoder.fit_transform(df_scheme_a['final_label'])
df_scheme_b['label_encoded'] = encoder.transform(df_scheme_b['final_label'])
df_scheme_c['label_encoded'] = encoder.transform(df_scheme_c['final_label'])

print(f"Classes: {encoder.classes_}")

print("\n" + "="*50)
print("RANDOM FOREST WITH BEST PARAMETERS")
print("="*50)

def random_forest_best(df, scheme_name):
    print(f"\n--- {scheme_name} ---")

    X = df.drop(['final_label', 'label_encoded'], axis=1).values
    y = df["label_encoded"].values

    print("Features shape:", X.shape)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=0.2,
        random_state=42,
        stratify=y
    )

    rf = RandomForestClassifier(
        n_estimators=200,
        max_depth=15,
        random_state=42,
        n_jobs=-1
    )

    rf.fit(X_train, y_train)

    y_pred = rf.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average="weighted")

    print(f"Accuracy: {acc:.4f}")
    print(f"F1 Score: {f1:.4f}")

    print(classification_report(y_test, y_pred, target_names=encoder.classes_))

    return acc, f1, rf

acc_a_RF_BOW, f1_a_RF_BOW, rf_a = random_forest_best(df_scheme_a, "Scheme A")
acc_b_RF_BOW, f1_b_RF_BOW, rf_b_RF_BOW = random_forest_best(df_scheme_b, "Scheme B")
acc_c_RF_BOW, f1_c_RF_BOW, rf_c = random_forest_best(df_scheme_c, "Scheme C")

print("\n" + "="*50)
print("FINAL RESULTS")
print("="*50)

print(f"Scheme A -> Accuracy: {acc_a_RF_BOW:.4f} | F1: {f1_a_RF_BOW:.4f}")
print(f"Scheme B -> Accuracy: {acc_b_RF_BOW:.4f} | F1: {f1_b_RF_BOW:.4f}")
print(f"Scheme C -> Accuracy: {acc_c_RF_BOW:.4f} | F1: {f1_c_RF_BOW:.4f}")

# Best by Accuracy
acc_list = [acc_a_RF_BOW, acc_b_RF_BOW, acc_c_RF_BOW]
f1_list = [f1_a_RF_BOW, f1_b_RF_BOW, f1_c_RF_BOW]
scheme_names = ["Scheme A", "Scheme B", "Scheme C"]

best_acc = max(acc_list)
best_acc_scheme = scheme_names[acc_list.index(best_acc)]

best_f1 = max(f1_list)
best_f1_scheme = scheme_names[f1_list.index(best_f1)]

print("\n" + "="*50)
print("BEST RESULTS")
print("="*50)

print(f"Best Accuracy: {best_acc_scheme} -> {best_acc:.4f} ({best_acc*100:.2f}%)")
print(f"Best F1 Score: {best_f1_scheme} -> {best_f1:.4f}")

LOADING DATA
Scheme A: (200, 509)
Scheme B: (200, 489)
Scheme C: (200, 384)

ALIGNING DATA
Using 200 rows

LABEL ENCODING
Classes: ['negative' 'neutral' 'positive']

RANDOM FOREST WITH BEST PARAMETERS

--- Scheme A ---
Features shape: (200, 509)
Accuracy: 0.6000
F1 Score: 0.4986
              precision    recall  f1-score   support

    negative       0.67      0.20      0.31        10
     neutral       0.00      0.00      0.00         7
    positive       0.59      0.96      0.73        23

    accuracy                           0.60        40
   macro avg       0.42      0.39      0.35        40
weighted avg       0.51      0.60      0.50        40


--- Scheme B ---
Features shape: (200, 489)
Accuracy: 0.6250
F1 Score: 0.5169
              precision    recall  f1-score   support

    negative       1.00      0.20      0.33        10
     neutral       0.00      0.00      0.00         7
    positive       0.61      1.00      0.75        23

    accuracy                           0.6

### Glove RANDOM FOREST

In [40]:
print("="*50)
print("LOADING GLOVE DATA")
print("="*50)

df_glove_a = pd.read_csv(r"D:\chrom download\glove_schema1.csv")
df_glove_b = pd.read_csv(r"D:\chrom download\glove_schema2.csv")
df_glove_c = pd.read_csv(r"D:\chrom download\glove_schema3.csv")

print(f"Scheme A: {df_glove_a.shape}")
print(f"Scheme B: {df_glove_b.shape}")
print(f"Scheme C: {df_glove_c.shape}")

def extract_embeddings(df):
    embeddings = []

    for emb in df["embedding"]:
        if isinstance(emb, str):
            vec = [float(x) for x in emb.split(",")]
            embeddings.append(vec)
        else:
            embeddings.append(emb)

    return np.array(embeddings)

print("\n" + "="*50)
print("ALIGNING DATA")
print("="*50)

emb_a = extract_embeddings(df_glove_a)
emb_b = extract_embeddings(df_glove_b)
emb_c = extract_embeddings(df_glove_c)

min_rows = min(len(emb_a), len(emb_b), len(emb_c), len(sample_df))
print(f"Using {min_rows} rows")

emb_a = emb_a[:min_rows]
emb_b = emb_b[:min_rows]
emb_c = emb_c[:min_rows]

df_truth_aligned = sample_df.iloc[:min_rows].copy()

print("\n" + "="*50)
print("LABEL ENCODING")
print("="*50)

encoder = LabelEncoder()
y = encoder.fit_transform(df_truth_aligned["final_label"])

print("Classes:", encoder.classes_)

print("\n" + "="*50)
print("RANDOM FOREST WITH BEST PARAMETERS")
print("="*50)

def random_forest_glove(X, y, scheme_name):
    print(f"\n--- {scheme_name} ---")

    X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)

    print("Features shape:", X.shape)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=0.2,
        random_state=42,
        stratify=y
    )

    rf = RandomForestClassifier(
        n_estimators=200,
        max_depth=15,
        random_state=42,
        n_jobs=-1
    )

    rf.fit(X_train, y_train)

    y_pred = rf.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average="weighted")

    print(f"Accuracy: {acc:.4f}")
    print(f"F1 Score: {f1:.4f}")

    print(classification_report(y_test, y_pred, target_names=encoder.classes_))

    return acc, f1, rf



acc_a_RF_GLOVE, f1_a_RF_GLOVE, rf_a = random_forest_glove(emb_a, y, "Scheme A")
acc_b_RF_GLOVE, f1_b_RF_GLOVE, rf_b = random_forest_glove(emb_b, y, "Scheme B")
acc_c_RF_GLOVE, f1_c_RF_GLOVE, rf_c = random_forest_glove(emb_c, y, "Scheme C")




print("\n" + "="*50)
print("FINAL RESULTS")
print("="*50)

print(f"Scheme A -> Accuracy: {acc_a_RF_GLOVE:.4f} | F1: {f1_a_RF_GLOVE:.4f}")
print(f"Scheme B -> Accuracy: {acc_b_RF_GLOVE:.4f} | F1: {f1_b_RF_GLOVE:.4f}")
print(f"Scheme C -> Accuracy: {acc_c_RF_GLOVE:.4f} | F1: {f1_c_RF_GLOVE:.4f}")

acc_list = [acc_a_RF_GLOVE, acc_b_RF_GLOVE, acc_c_RF_GLOVE]
f1_list = [f1_a_RF_GLOVE, f1_b_RF_GLOVE, f1_c_RF_GLOVE]
scheme_names = ["Scheme A", "Scheme B", "Scheme C"]

best_acc = max(acc_list)
best_acc_scheme = scheme_names[acc_list.index(best_acc)]

best_f1 = max(f1_list)
best_f1_scheme = scheme_names[f1_list.index(best_f1)]

print("\n" + "="*50)
print("BEST RESULTS")
print("="*50)

print(f"Best Accuracy: {best_acc_scheme} -> {best_acc:.4f} ({best_acc*100:.2f}%)")
print(f"Best F1 Score: {best_f1_scheme} -> {best_f1:.4f}")

LOADING GLOVE DATA
Scheme A: (200, 18)
Scheme B: (200, 18)
Scheme C: (200, 18)

ALIGNING DATA
Using 200 rows

LABEL ENCODING
Classes: ['negative' 'neutral' 'positive']

RANDOM FOREST WITH BEST PARAMETERS

--- Scheme A ---
Features shape: (200, 100)
Accuracy: 0.7000
F1 Score: 0.6861
              precision    recall  f1-score   support

    negative       0.83      0.50      0.62        10
     neutral       0.60      0.43      0.50         7
    positive       0.69      0.87      0.77        23

    accuracy                           0.70        40
   macro avg       0.71      0.60      0.63        40
weighted avg       0.71      0.70      0.69        40


--- Scheme B ---
Features shape: (200, 100)
Accuracy: 0.7500
F1 Score: 0.7388
              precision    recall  f1-score   support

    negative       1.00      0.50      0.67        10
     neutral       0.67      0.57      0.62         7
    positive       0.72      0.91      0.81        23

    accuracy                           

In [41]:
!pip install nbformat

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: C:\Program Files\Python313\python.exe -m pip install --upgrade pip


In [42]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# =========================
# LABELS
# =========================
bow_labels = ["Scheme A", "Scheme B", "Scheme C"]
glove_labels = ["Scheme A", "Scheme B", "Scheme C"]

# =========================
# BOw VALUES (Random Forest BoW)
# =========================
bow_acc = [acc_a_RF_BOW, acc_b_RF_BOW, acc_c_RF_BOW]
bow_f1  = [f1_a_RF_BOW, f1_b_RF_BOW, f1_c_RF_BOW]

# =========================
# GLOVE VALUES (Random Forest GloVe)
# =========================
glove_acc = [acc_a_RF_GLOVE, acc_b_RF_GLOVE, acc_c_RF_GLOVE]
glove_f1  = [f1_a_RF_GLOVE, f1_b_RF_GLOVE, f1_c_RF_GLOVE]

# =========================
# FIGURE
# =========================
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        "🔥 BoW Accuracy",
        "🔥 GloVe Accuracy",
        "📊 BoW F1 Score",
        "📊 GloVe F1 Score"
    ),
    vertical_spacing=0.15
)

# =========================
# BOw ACC
# =========================
fig.add_trace(go.Bar(
    x=bow_labels,
    y=bow_acc,
    text=[f"{x:.2%}" for x in bow_acc],
    textposition="outside",
    marker_color="#636EFA"
), row=1, col=1)

# =========================
# GLOVE ACC
# =========================
fig.add_trace(go.Bar(
    x=glove_labels,
    y=glove_acc,
    text=[f"{x:.2%}" for x in glove_acc],
    textposition="outside",
    marker_color="#00CC96"
), row=1, col=2)

# =========================
# BOw F1
# =========================
fig.add_trace(go.Bar(
    x=bow_labels,
    y=bow_f1,
    text=[f"{x:.2%}" for x in bow_f1],
    textposition="outside",
    marker_color="#FFA15A"
), row=2, col=1)

# =========================
# GLOVE F1
# =========================
fig.add_trace(go.Bar(
    x=glove_labels,
    y=glove_f1,
    text=[f"{x:.2%}" for x in glove_f1],
    textposition="outside",
    marker_color="#AB63FA"
), row=2, col=2)

# =========================
# STYLE (PRO DARK DASHBOARD)
# =========================
fig.update_layout(
    template="plotly_dark",
    height=850,
    width=1200,
    title="🏆 BoW vs GloVe (Random Forest Performance)",
    showlegend=False,
    font=dict(size=13)
)

fig.update_yaxes(range=[0, 1])

fig.show()

# Naive Bias BOW

In [43]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.model_selection import train_test_split
import pandas as pd

print("="*50)
print("LOADING BOW DATA")
print("="*50)

df_bow_a = pd.read_csv(r"D:\chrom download\bow_scheme_a.csv")
df_bow_b = pd.read_csv(r"D:\chrom download\bow_scheme_b.csv")
df_bow_c = pd.read_csv(r"D:\chrom download\bow_scheme_c.csv")

print(df_bow_a.shape)
print(df_bow_b.shape)
print(df_bow_c.shape)

def naive_bayes_bow(df, y, scheme_name, alpha=0.5):
    print(f"\n--- {scheme_name} ---")

    X = df.values

    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=0.2,
        random_state=42,
        stratify=y
    )

    nb = MultinomialNB(alpha=alpha)
    nb.fit(X_train, y_train)

    y_pred = nb.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average="weighted")

    print(f"Accuracy: {acc:.4f}")
    print(f"F1 Score: {f1:.4f}")
    print(classification_report(y_test, y_pred))

    return acc, f1, nb



nb_acc_a_BOW, nb_f1_a_BOW, nb_model_a = naive_bayes_bow(df_bow_a, y, "BoW Scheme A", alpha=0.5)
nb_acc_b_BOW, nb_f1_b_BOW, nb_model_b = naive_bayes_bow(df_bow_b, y, "BoW Scheme B", alpha=0.5)
nb_acc_c_BOW, nb_f1_c_BOW, nb_model_c = naive_bayes_bow(df_bow_c, y, "BoW Scheme C", alpha=0.7)


print("\n" + "="*50)
print("FINAL RESULTS")
print("="*50)

print(f"Scheme A -> Accuracy: {nb_acc_a_BOW:.4f} | F1: {nb_f1_a_BOW:.4f}")
print(f"Scheme B -> Accuracy: {nb_acc_b_BOW:.4f} | F1: {nb_f1_b_BOW:.4f}")
print(f"Scheme C -> Accuracy: {nb_acc_c_BOW:.4f} | F1: {nb_f1_c_BOW:.4f}")

acc_list = [nb_acc_a_BOW, nb_acc_b_BOW, nb_acc_c_BOW]
f1_list = [nb_f1_a_BOW, nb_f1_b_BOW, nb_f1_c_BOW]
scheme_names = ["Scheme A", "Scheme B", "Scheme C"]

best_acc = max(acc_list)
best_acc_scheme = scheme_names[acc_list.index(best_acc)]

best_f1 = max(f1_list)
best_f1_scheme = scheme_names[f1_list.index(best_f1)]

print("\n" + "="*50)
print("BEST RESULTS")
print("="*50)

print(f"Best Accuracy: {best_acc_scheme} -> {best_acc:.4f} ({best_acc*100:.2f}%)")
print(f"Best F1 Score: {best_f1_scheme} -> {best_f1:.4f}")

LOADING BOW DATA
(200, 510)
(200, 490)
(200, 385)

--- BoW Scheme A ---
Accuracy: 0.5750
F1 Score: 0.5499
              precision    recall  f1-score   support

           0       0.56      0.50      0.53        10
           1       0.33      0.14      0.20         7
           2       0.61      0.74      0.67        23

    accuracy                           0.57        40
   macro avg       0.50      0.46      0.46        40
weighted avg       0.55      0.57      0.55        40


--- BoW Scheme B ---
Accuracy: 0.6000
F1 Score: 0.5760
              precision    recall  f1-score   support

           0       0.60      0.60      0.60        10
           1       0.33      0.14      0.20         7
           2       0.63      0.74      0.68        23

    accuracy                           0.60        40
   macro avg       0.52      0.49      0.49        40
weighted avg       0.57      0.60      0.58        40


--- BoW Scheme C ---
Accuracy: 0.6000
F1 Score: 0.5610
              precis

### Glove naive bias

In [44]:
from sklearn.naive_bayes import GaussianNB
import numpy as np

print("="*50)
print("LOADING GLOVE DATA")
print("="*50)

df_glove_a = pd.read_csv(r"D:\chrom download\glove_schema1.csv")
df_glove_b = pd.read_csv(r"D:\chrom download\glove_schema2.csv")
df_glove_c = pd.read_csv(r"D:\chrom download\glove_schema3.csv")

def process_embeddings(df):
    embeddings = df["embedding"].apply(lambda x: np.fromstring(x, sep=","))
    return np.vstack(embeddings)

emb_a = process_embeddings(df_glove_a)
emb_b = process_embeddings(df_glove_b)
emb_c = process_embeddings(df_glove_c)


def naive_bayes_glove(X, y, scheme_name, smoothing=1e-9):
    print(f"\n--- {scheme_name} ---")

    X = np.nan_to_num(X)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=0.2,
        random_state=42,
        stratify=y
    )

    nb = GaussianNB(var_smoothing=smoothing)
    nb.fit(X_train, y_train)

    y_pred = nb.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average="weighted")

    print(f"Accuracy: {acc:.4f}")
    print(f"F1 Score: {f1:.4f}")
    print(classification_report(y_test, y_pred))

    return acc, f1, nb


g_nb_acc_a, g_nb_f1_a, g_nb_model_a = naive_bayes_glove(emb_a, y, "GloVe Scheme A", 1e-7)
g_nb_acc_b, g_nb_f1_b, g_nb_model_b = naive_bayes_glove(emb_b, y, "GloVe Scheme B", 1e-9)
g_nb_acc_c, g_nb_f1_c, g_nb_model_c = naive_bayes_glove(emb_c, y, "GloVe Scheme C", 1e-9)


print("\n" + "="*50)
print("FINAL RESULTS")
print("="*50)

print(f"Scheme A -> Accuracy: {g_nb_acc_a:.4f} | F1: {g_nb_f1_a:.4f}")
print(f"Scheme B -> Accuracy: {g_nb_acc_b:.4f} | F1: {g_nb_f1_b:.4f}")
print(f"Scheme C -> Accuracy: {g_nb_acc_c:.4f} | F1: {g_nb_f1_c:.4f}")

acc_list = [g_nb_acc_a, g_nb_acc_b, g_nb_acc_c]
f1_list = [g_nb_f1_a, g_nb_f1_b, g_nb_f1_c]
scheme_names = ["Scheme A", "Scheme B", "Scheme C"]

best_acc = max(acc_list)
best_acc_scheme = scheme_names[acc_list.index(best_acc)]

best_f1 = max(f1_list)
best_f1_scheme = scheme_names[f1_list.index(best_f1)]

print("\n" + "="*50)
print("BEST RESULTS")
print("="*50)

print(f"Best Accuracy: {best_acc_scheme} -> {best_acc:.4f} ({best_acc*100:.2f}%)")
print(f"Best F1 Score: {best_f1_scheme} -> {best_f1:.4f}")


LOADING GLOVE DATA

--- GloVe Scheme A ---
Accuracy: 0.5500
F1 Score: 0.5384
              precision    recall  f1-score   support

           0       0.47      0.90      0.62        10
           1       0.44      0.57      0.50         7
           2       0.75      0.39      0.51        23

    accuracy                           0.55        40
   macro avg       0.56      0.62      0.54        40
weighted avg       0.63      0.55      0.54        40


--- GloVe Scheme B ---
Accuracy: 0.6250
F1 Score: 0.6274
              precision    recall  f1-score   support

           0       0.60      0.60      0.60        10
           1       0.50      0.71      0.59         7
           2       0.70      0.61      0.65        23

    accuracy                           0.62        40
   macro avg       0.60      0.64      0.61        40
weighted avg       0.64      0.62      0.63        40


--- GloVe Scheme C ---
Accuracy: 0.4750
F1 Score: 0.4420
              precision    recall  f1-score  

In [45]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# =========================
# LABELS
# =========================
bow_labels = ["Scheme A", "Scheme B", "Scheme C"]
glove_labels = ["Scheme A", "Scheme B", "Scheme C"]

# =========================
# BoW RESULTS (Naive Bayes)
# =========================
bow_acc = [nb_acc_a_BOW, nb_acc_b_BOW, nb_acc_c_BOW]
bow_f1  = [nb_f1_a_BOW, nb_f1_b_BOW, nb_f1_c_BOW]

# =========================
# GLOVE RESULTS (Naive Bayes)
# =========================
glove_acc = [g_nb_acc_a, g_nb_acc_b, g_nb_acc_c]
glove_f1  = [g_nb_f1_a, g_nb_f1_b, g_nb_f1_c]

# =========================
# FIGURE
# =========================
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        "Naive Bayes - BoW Accuracy",
        "Naive Bayes - GloVe Accuracy",
        "Naive Bayes - BoW F1 Score",
        "Naive Bayes - GloVe F1 Score"
    )
)

# =========================
# ROW 1 - ACCURACY
# =========================
fig.add_trace(go.Bar(
    x=bow_labels,
    y=bow_acc,
    text=[f"{v:.2%}" for v in bow_acc],
    textposition="outside"
), row=1, col=1)

fig.add_trace(go.Bar(
    x=glove_labels,
    y=glove_acc,
    text=[f"{v:.2%}" for v in glove_acc],
    textposition="outside"
), row=1, col=2)

# =========================
# ROW 2 - F1 SCORE
# =========================
fig.add_trace(go.Bar(
    x=bow_labels,
    y=bow_f1,
    text=[f"{v:.2%}" for v in bow_f1],
    textposition="outside"
), row=2, col=1)

fig.add_trace(go.Bar(
    x=glove_labels,
    y=glove_f1,
    text=[f"{v:.2%}" for v in glove_f1],
    textposition="outside"
), row=2, col=2)

# =========================
# STYLE (DARK PROFESSIONAL)
# =========================
fig.update_layout(
    template="plotly_dark",
    height=800,
    width=1200,
    title="Naive Bayes Performance Dashboard (BoW vs GloVe)",
    showlegend=False
)

fig.update_yaxes(range=[0, 1])

fig.show()

# Final Comparison

In [46]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# =========================
# DATA
# =========================
model_names = np.array([
    "SentiWordNet",
    "Bing Liu",

    "BoW RF A", "BoW RF B", "BoW RF C",
    "GloVe RF A", "GloVe RF B", "GloVe RF C",

    "BoW NB A", "BoW NB B", "BoW NB C",
    "GloVe NB A", "GloVe NB B", "GloVe NB C"
])

acc_values = np.array([
    acc_sent,
    acc_bing,

    acc_a_RF_BOW, acc_b_RF_BOW, acc_c_RF_BOW,
    acc_a_RF_GLOVE, acc_b_RF_GLOVE, acc_c_RF_GLOVE,

    nb_acc_a_BOW, nb_acc_b_BOW, nb_acc_c_BOW,
    g_nb_acc_a, g_nb_acc_b, g_nb_acc_c
])

f1_values = np.array([
    f1_sent,
    f1_bing,

    f1_a_RF_BOW, f1_b_RF_BOW, f1_c_RF_BOW,
    f1_a_RF_GLOVE, f1_b_RF_GLOVE, f1_c_RF_GLOVE,

    nb_f1_a_BOW, nb_f1_b_BOW, nb_f1_c_BOW,
    g_nb_f1_a, g_nb_f1_b, g_nb_f1_c
])

# =========================
# SORT ACCURACY
# =========================
acc_idx = np.argsort(acc_values)
acc_names_sorted = model_names[acc_idx]
acc_sorted = acc_values[acc_idx]

# =========================
# SORT F1
# =========================
f1_idx = np.argsort(f1_values)
f1_names_sorted = model_names[f1_idx]
f1_sorted = f1_values[f1_idx]

# =========================
# COLORS (highlight best)
# =========================
acc_colors = ["#636EFA"] * len(acc_sorted)
f1_colors = ["#FFA15A"] * len(f1_sorted)

acc_colors[-1] = "gold"
f1_colors[-1] = "gold"

# =========================
# FIGURE (smaller + clean layout)
# =========================
fig = make_subplots(
    rows=2, cols=1,
    subplot_titles=(
        "🔥 Accuracy Ranking (Sorted Low → High)",
        "📊 F1 Score Ranking (Sorted Low → High)"
    ),
    vertical_spacing=0.20
)

# =========================
# ACCURACY
# =========================
fig.add_trace(
    go.Bar(
        x=acc_names_sorted,
        y=acc_sorted,
        text=[f"{v:.2%}" for v in acc_sorted],
        textposition="outside",
        marker_color=acc_colors
    ),
    row=1, col=1
)

# =========================
# F1 SCORE
# =========================
fig.add_trace(
    go.Bar(
        x=f1_names_sorted,
        y=f1_sorted,
        text=[f"{v:.2%}" for v in f1_sorted],
        textposition="outside",
        marker_color=f1_colors
    ),
    row=2, col=1
)

# =========================
# STYLE (FULL SCREEN CLEAN)
# =========================
fig.update_layout(
    template="plotly_dark",
    height=850,
    width=1400,   # 👈 أصغر عشان يناسب الشاشة
    title="🏆 Sentiment Models Ranking (Accuracy & F1 Sorted)",
    showlegend=False,
    font=dict(size=12),
    margin=dict(l=40, r=40, t=80, b=40)
)

fig.update_yaxes(range=[0, 1])

fig.update_xaxes(tickangle=-45)

fig.show()

# thanks

In [47]:
def random_forest_glove(X, y, scheme_name):
    print(f"\n--- {scheme_name} ---")

    X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=0.2,
        random_state=42,
        stratify=y
    )

    rf = RandomForestClassifier(
        n_estimators=200,
        max_depth=15,
        random_state=42,
        n_jobs=-1
    )

    rf.fit(X_train, y_train)
    y_pred = rf.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average="weighted")

    print(f"Accuracy: {acc:.4f}")
    print(f"F1 Score: {f1:.4f}")

    return acc, f1, rf, X_test, y_test

In [48]:
acc_b_RF_GLOVE, f1_b_RF_GLOVE, rf_b, X_test_b, y_test_b = random_forest_glove(
    emb_b, y, "Scheme B"
)


--- Scheme B ---
Accuracy: 0.7500
F1 Score: 0.7388


In [49]:
from sklearn.metrics import confusion_matrix
import plotly.express as px

# prediction on exact test split
y_pred_b = rf_b.predict(X_test_b)

cm = confusion_matrix(y_test_b, y_pred_b, normalize="true")

fig = px.imshow(
    cm,
    x=encoder.classes_,
    y=encoder.classes_,
    text_auto=True,
    labels=dict(
        x="Predicted",
        y="Actual",
        color="Count"
    ),
    title="🔥 Confusion Matrix - Random Forest GloVe Schema 2",
    template="plotly_dark",
    aspect="auto"
)

fig.update_layout(
    width=700,
    height=600,
    font=dict(size=14)
)

fig.show()

In [50]:
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import label_binarize
import numpy as np

# =========================
# PREDICT PROBABILITIES
# =========================
y_proba_b = rf_b.predict_proba(X_test_b)

# =========================
# BINARIZE TRUE LABELS
# =========================
classes = np.unique(y_test_b)
y_test_bin = label_binarize(y_test_b, classes=classes)

# =========================
# MULTI-CLASS ROC AUC
# =========================
roc_auc_b = roc_auc_score(
    y_test_bin,
    y_proba_b,
    multi_class="ovr",
    average="weighted"
)

print(f"🔥 ROC-AUC (RF GloVe Schema 2): {roc_auc_b:.4f}")

🔥 ROC-AUC (RF GloVe Schema 2): 0.8152


In [51]:
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import label_binarize
import plotly.graph_objects as go
import numpy as np

# =========================
# PREPARE SCHEMA 2 DATA
# =========================
X = np.nan_to_num(emb_b, nan=0.0, posinf=0.0, neginf=0.0)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# =========================
# PREDICT PROBABILITIES
# =========================
y_score = rf_b.predict_proba(X_test)

# =========================
# BINARIZE LABELS
# =========================
n_classes = len(encoder.classes_)
y_test_bin = label_binarize(y_test, classes=np.arange(n_classes))

# =========================
# CREATE ROC FIGURE
# =========================
fig = go.Figure()

for i, class_name in enumerate(encoder.classes_):
    fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_score[:, i])
    roc_auc = auc(fpr, tpr)

    fig.add_trace(go.Scatter(
        x=fpr,
        y=tpr,
        mode="lines",
        name=f"{class_name} (AUC = {roc_auc:.3f})",
        line=dict(width=4)
    ))

# Random baseline
fig.add_trace(go.Scatter(
    x=[0, 1],
    y=[0, 1],
    mode="lines",
    name="Random Guess",
    line=dict(dash="dash")
))

# =========================
# STYLE
# =========================
fig.update_layout(
    template="plotly_dark",
    title="🔥 ROC Curve - Random Forest GloVe Schema 2",
    xaxis_title="False Positive Rate",
    yaxis_title="True Positive Rate",
    width=1000,
    height=700,
    font=dict(size=14),
    legend_title="Classes"
)

fig.show()

# optamization

In [52]:
# import numpy as np
# from sklearn.ensemble import RandomForestClassifier
# from sklearn.model_selection import train_test_split, GridSearchCV
# from sklearn.metrics import accuracy_score, f1_score, classification_report

# X = emb_b
# X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)

# X_train, X_test, y_train, y_test = train_test_split(
#     X, y,
#     test_size=0.2,
#     random_state=42,
#     stratify=y
# )
# param_grid = {
#     "n_estimators": [100, 200, 300],
#     "max_depth": [10, 15, 20, None],
#     "min_samples_split": [2, 5, 10],
#     "min_samples_leaf": [1, 2, 4]
# }

# rf = RandomForestClassifier(random_state=42, n_jobs=-1)

# grid = GridSearchCV(
#     estimator=rf,
#     param_grid=param_grid,
#     scoring="f1_weighted",
#     cv=3,
#     verbose=2
# )

# grid.fit(X_train, y_train)

# print("Best Params:", grid.best_params_)
# best_model = grid.best_estimator_

# y_pred = best_model.predict(X_test)

# acc_opt = accuracy_score(y_test, y_pred)
# f1_opt = f1_score(y_test, y_pred, average="weighted")

# print("="*50)
# print("OPTIMIZED MODEL RESULTS (GLOVE SCHEMA 2)")
# print("="*50)

# print("Accuracy:", acc_opt)
# print("F1 Score:", f1_opt)

# print("\nClassification Report:\n")
# print(classification_report(y_test, y_pred))

In [53]:
# print("Accuracy:", acc_opt)
# # print("F1 Score:", f1_opt)


In [ ]:
# import numpy as np
# import matplotlib.pyplot as plt
# import seaborn as sns
# from sklearn.metrics import confusion_matrix

# # =========================
# # PREDICTIONS
# # =========================
# y_pred = best_model.predict(X_test)

# # =========================
# # CONFUSION MATRIX
# # =========================
# cm = confusion_matrix(y_test, y_pred)

# classes = encoder.classes_  # positive / negative / natural

# # =========================
# # PLOT
# # =========================
# plt.figure(figsize=(6, 5))
# sns.heatmap(
#     cm,
#     annot=True,
#     fmt="d",
#     cmap="Blues",
#     xticklabels=classes,
#     yticklabels=classes
# )

# plt.title("Confusion Matrix - Optimized Random Forest (GloVe Schema 2)")
# plt.xlabel("Predicted Label")
# plt.ylabel("True Label")
# plt.show()

NameError: name 'best_model' is not defined

In [ ]:
# print("="*50)
# print("COMPARISON BEFORE vs AFTER OPTIMIZATION")
# print("="*50)

# print(f"Before Optimization F1: {f1_b_RF_GLOVE:.4f}")
# print(f"After Optimization  F1: {f1_opt:.4f}")

# improvement = (f1_opt - f1_b_RF_GLOVE) / f1_b_RF_GLOVE * 100

# print(f"\nImprovement: {improvement:.2f}%")

COMPARISON BEFORE vs AFTER OPTIMIZATION
Before Optimization F1: 0.7388
After Optimization  F1: 0.6703

Improvement: -9.27%


In [ ]:
# import numpy as np
# from sklearn.ensemble import RandomForestClassifier
# from sklearn.model_selection import train_test_split, RandomizedSearchCV
# from sklearn.metrics import accuracy_score, f1_score, classification_report

# # =========================
# # DATA (GLOVE SCHEMA 2)
# # =========================
# X = np.nan_to_num(emb_b, nan=0.0, posinf=0.0, neginf=0.0)

# X_train, X_test, y_train, y_test = train_test_split(
#     X, y,
#     test_size=0.2,
#     random_state=42,
#     stratify=y
# )

# # =========================
# # PARAM DISTRIBUTION
# # =========================
# param_dist = {
#     "n_estimators": np.arange(100, 800, 50),
#     "max_depth": [None] + list(np.arange(5, 50, 5)),
#     "min_samples_split": np.arange(2, 20),
#     "min_samples_leaf": np.arange(1, 10),
#     "max_features": ["sqrt", "log2", None]
# }

# # =========================
# # BASE MODEL
# # =========================
# rf = RandomForestClassifier(
#     random_state=42,
#     n_jobs=-1,
#     class_weight="balanced"   # 🔥 مهم لتحسين F1
# )

# # =========================
# # RANDOM SEARCH
# # =========================
# random_search = RandomizedSearchCV(
#     estimator=rf,
#     param_distributions=param_dist,
#     n_iter=50,               # عدد التجارب
#     scoring="f1_weighted",   # مهم جدًا
#     cv=3,
#     verbose=2,
#     random_state=42,
#     n_jobs=-1
# )

# random_search.fit(X_train, y_train)

# # =========================
# # BEST MODEL
# # =========================
# best_model = random_search.best_estimator_

# print("="*50)
# print("BEST PARAMETERS")
# print("="*50)
# print(random_search.best_params_)

# # =========================
# # EVALUATION
# # =========================
# y_pred = best_model.predict(X_test)

# acc_opt = accuracy_score(y_test, y_pred)
# f1_opt = f1_score(y_test, y_pred, average="weighted")

# print("\n" + "="*50)
# print("OPTIMIZED MODEL RESULTS")
# print("="*50)

# print("Accuracy:", acc_opt)
# print("F1 Score:", f1_opt)

# print("\nClassification Report:\n")
# print(classification_report(y_test, y_pred))

Fitting 3 folds for each of 50 candidates, totalling 150 fits
BEST PARAMETERS
{'n_estimators': 250, 'min_samples_split': 4, 'min_samples_leaf': 5, 'max_features': 'log2', 'max_depth': 30}

OPTIMIZED MODEL RESULTS
Accuracy: 0.625
F1 Score: 0.6268428470754053

Classification Report:

              precision    recall  f1-score   support

           0       0.58      0.70      0.64        10
           1       0.50      0.57      0.53         7
           2       0.70      0.61      0.65        23

    accuracy                           0.62        40
   macro avg       0.59      0.63      0.61        40
weighted avg       0.64      0.62      0.63        40



In [55]:
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score, f1_score, classification_report

X = emb_b
X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)
param_grid = {
    "n_estimators": [200, 300, 400],
    "max_depth": [10, 15, 20],
    "min_samples_leaf": [1, 2, 3],
    "class_weight": ["balanced"]
}

rf = RandomForestClassifier(random_state=42, n_jobs=-1,class_weight="balanced")

grid = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    scoring="f1_weighted",
    cv=5,
    verbose=2
)

grid.fit(X_train, y_train)

print("Best Params:", grid.best_params_)
best_model = grid.best_estimator_

y_pred = best_model.predict(X_test)

acc_opt = accuracy_score(y_test, y_pred)
f1_opt = f1_score(y_test, y_pred, average="weighted")

print("="*50)
print("OPTIMIZED MODEL RESULTS (GLOVE SCHEMA 2)")
print("="*50)

print("Accuracy:", acc_opt)
print("F1 Score:", f1_opt)

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

Fitting 5 folds for each of 27 candidates, totalling 135 fits
[CV] END class_weight=balanced, max_depth=10, min_samples_leaf=1, n_estimators=200; total time=   0.4s
[CV] END class_weight=balanced, max_depth=10, min_samples_leaf=1, n_estimators=200; total time=   0.3s
[CV] END class_weight=balanced, max_depth=10, min_samples_leaf=1, n_estimators=200; total time=   0.2s
[CV] END class_weight=balanced, max_depth=10, min_samples_leaf=1, n_estimators=200; total time=   0.2s
[CV] END class_weight=balanced, max_depth=10, min_samples_leaf=1, n_estimators=200; total time=   0.2s
[CV] END class_weight=balanced, max_depth=10, min_samples_leaf=1, n_estimators=300; total time=   0.3s
[CV] END class_weight=balanced, max_depth=10, min_samples_leaf=1, n_estimators=300; total time=   0.5s
[CV] END class_weight=balanced, max_depth=10, min_samples_leaf=1, n_estimators=300; total time=   0.4s
[CV] END class_weight=balanced, max_depth=10, min_samples_leaf=1, n_estimators=300; total time=   0.4s
[CV] END cl

In [ ]:
print("Accuracy:", acc_opt)
print("F1 Score:", f1_opt)

Accuracy: 0.75
F1 Score: 0.7387820512820513


In [ ]:
print("="*50)
print("COMPARISON BEFORE vs AFTER OPTIMIZATION")
print("="*50)

print(f"Before Optimization F1: {f1_b_RF_GLOVE:.4f}")
print(f"After Optimization  F1: {f1_opt:.4f}")

improvement = (f1_opt - f1_b_RF_GLOVE) / f1_b_RF_GLOVE * 100

print(f"\nImprovement: {improvement:.2f}%")

COMPARISON BEFORE vs AFTER OPTIMIZATION
Before Optimization F1: 0.7388
After Optimization  F1: 0.7388

Improvement: 0.00%


In [ ]:
# import numpy as np
# from sklearn.ensemble import RandomForestClassifier
# from sklearn.model_selection import train_test_split, GridSearchCV
# from sklearn.metrics import accuracy_score, f1_score, classification_report

# X = emb_b
# X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)

# X_train, X_test, y_train, y_test = train_test_split(
#     X, y,
#     test_size=0.2,
#     random_state=42,
#     stratify=y
# )
# param_dist = {
#     "n_estimators": [200, 300, 500, 800],
#     "max_depth": [10, 20, 30, None],
#     "min_samples_split": [2, 5, 10, 20],
#     "min_samples_leaf": [1, 2, 4, 8],
#     "max_features": ["sqrt", "log2", None],
#     "bootstrap": [True, False],
#     "criterion": ["gini", "entropy"]
# }

# rf = RandomForestClassifier(random_state=42, n_jobs=-1)

# grid = GridSearchCV(
#     estimator=rf,
#     param_grid=param_grid,
#     scoring="f1_weighted",
#     cv=5,
#     verbose=2
# )

# grid.fit(X_train, y_train)

# print("Best Params:", grid.best_params_)
# best_model = grid.best_estimator_

# y_pred = best_model.predict(X_test)

# acc_opt = accuracy_score(y_test, y_pred)
# f1_opt = f1_score(y_test, y_pred, average="weighted")

# print("="*50)
# print("OPTIMIZED MODEL RESULTS (GLOVE SCHEMA 2)")
# # print("="*50)

# print("Accuracy:", acc_opt)
# print("F1 Score:", f1_opt)

# print("\nClassification Report:\n")
# print(classification_report(y_test, y_pred))

Fitting 5 folds for each of 108 candidates, totalling 540 fits
[CV] END max_depth=10, min_samples_leaf=1, min_samples_split=2, n_estimators=100; total time=   0.0s
[CV] END max_depth=10, min_samples_leaf=1, min_samples_split=2, n_estimators=100; total time=   0.0s
[CV] END max_depth=10, min_samples_leaf=1, min_samples_split=2, n_estimators=100; total time=   0.0s
[CV] END max_depth=10, min_samples_leaf=1, min_samples_split=2, n_estimators=100; total time=   0.1s
[CV] END max_depth=10, min_samples_leaf=1, min_samples_split=2, n_estimators=100; total time=   0.0s
[CV] END max_depth=10, min_samples_leaf=1, min_samples_split=2, n_estimators=200; total time=   0.2s
[CV] END max_depth=10, min_samples_leaf=1, min_samples_split=2, n_estimators=200; total time=   0.2s
[CV] END max_depth=10, min_samples_leaf=1, min_samples_split=2, n_estimators=200; total time=   0.2s
[CV] END max_depth=10, min_samples_leaf=1, min_samples_split=2, n_estimators=200; total time=   0.2s
[CV] END max_depth=10, min_s

In [ ]:
# print("Accuracy:", acc_opt)
# print("F1 Score:", f1_opt)

Accuracy: 0.75
F1 Score: 0.7387820512820513


In [ ]:
# import numpy as np
# from sklearn.model_selection import train_test_split, GridSearchCV
# from sklearn.preprocessing import StandardScaler
# from sklearn.decomposition import PCA
# from sklearn.ensemble import RandomForestClassifier
# from sklearn.metrics import accuracy_score, f1_score, classification_report
# from sklearn.pipeline import Pipeline

# # =========================
# # 1. DATA
# # =========================
# X = emb_b  # GloVe Schema 2
# X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)

# # =========================
# # 2. TRAIN TEST SPLIT
# # =========================
# X_train, X_test, y_train, y_test = train_test_split(
#     X, y,
#     test_size=0.2,
#     random_state=42,
#     stratify=y
# )

# # =========================
# # 3. PIPELINE (Scaler → PCA → RF)
# # =========================
# pipe = Pipeline([
#     ("scaler", StandardScaler()),
#     ("pca", PCA()),
#     ("rf", RandomForestClassifier(random_state=42, n_jobs=-1))
# ])

# # =========================
# # 4. PARAM GRID
# # =========================
# param_grid = {
#     # PCA tuning
#     "pca__n_components": [50, 100, 150],

#     # Random Forest tuning
#     "rf__n_estimators": [300, 500, 800],
#     "rf__max_depth": [10, 20, 30, None],
#     "rf__min_samples_split": [2, 5, 10],
#     "rf__min_samples_leaf": [1, 2, 4],
#     "rf__max_features": ["sqrt", "log2"]
# }

# # =========================
# # 5. GRID SEARCH
# # =========================
# grid = GridSearchCV(
#     estimator=pipe,
#     param_grid=param_grid,
#     scoring="f1_weighted",
#     cv=3,
#     verbose=2,
#     n_jobs=-1
# )

# grid.fit(X_train, y_train)

# # =========================
# # 6. BEST MODEL
# # =========================
# best_model = grid.best_estimator_

# print("\n" + "="*50)
# print("BEST PARAMETERS")
# print("="*50)
# print(grid.best_params_)

# # =========================
# # 7. EVALUATION
# # =========================
# y_pred = best_model.predict(X_test)

# acc = accuracy_score(y_test, y_pred)
# f1 = f1_score(y_test, y_pred, average="weighted")

# print("\n" + "="*50)
# print("OPTIMIZED RESULTS (PCA + GRID SEARCH + RF)")
# print("="*50)

# print(f"Accuracy: {acc:.4f}")
# print(f"F1 Score: {f1:.4f}")

# print("\nClassification Report:\n")
# print(classification_report(y_test, y_pred))

NameError: name 'emb_b' is not defined

In [ ]:
misclassified

In [ ]:
import pandas as pd

def categorize_errors(row):
    text = row['review_text'].lower()
    
    # Pattern 1: Sarcasm/Irony
    if "gotta love" in text or "i love... sadly" in text:
        return "Sarcasm/Irony"
    
    # Pattern 2: Contrastive Adjectives (The "But" Pivot)
    if "but" in text or "tho" in text or "although" in text:
        return "Contrastive Logic"
    
    # Pattern 3: Gaming Slang/Memes
    memes = ["try fingers", "super earth", "monster my hunter"]
    if any(m in text for m in memes):
        return "Domain Slang/Memes"
    
    # Pattern 4: Length/Noise
    if row['review_length'] > 500:
        return "Long-form Noise"
    
    return "Uncategorized"

# Assuming 'errors_df' is your table above
misclassified['failure_reason'] = misclassified.apply(categorize_errors, axis=1)

# Summary for your report
summary = misclassified['failure_reason'].value_counts()
print(summary)

In [ ]:
# visualization using plotly
import matplotlib.pyplot as plt 
plt.figure(figsize=(8,5))
summary.plot(kind='bar', color=['orange', 'blue', 'green', 'red'])  
plt.title('Error Categorization of Misclassified Reviews')
plt.xlabel('Failure Reason')
plt.ylabel('Number of Reviews')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()